# Config Settings Rag

In [1]:
# ============================================================
# CELL 1 — RAG CONFIG (Multi-Source Folder Ingestion)
# ============================================================

from __future__ import annotations
import os
import re
import json
import hashlib
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict, Optional

# ----------------------------
# 1) Project / Unit Settings
# ----------------------------
UNIT_CODE = "MBAI5004"            # used for naming DB + outputs
SOURCES_DIR = Path("inputs/sources") / UNIT_CODE  # folder containing pdf/epub sources
OUTPUT_ROOT = Path("data_output") / UNIT_CODE

# Optional: unit outline path (used later to generate per-week topic queries)
UNIT_OUTLINE_PATH = Path("MBAI5004 AI, Ethics and Governance 04112025.docx")

# ----------------------------
# 2) File Discovery Settings
# ----------------------------
ALLOWED_EXTS = {".pdf", ".epub"}
PREFERRED_EXT_ORDER = [".epub", ".pdf"]  # if both exist for same work_id, pick epub
RECURSIVE = True

# ----------------------------
# 3) Vector DB Settings
# ----------------------------
CHROMA_PERSIST_DIR = OUTPUT_ROOT / "chroma_db"
CHROMA_COLLECTION_NAME = f"{UNIT_CODE}_multi_source"

REBUILD_INDEX = False  # True = wipe and rebuild; False = reuse if exists (you'll implement append/upsert later)

# ----------------------------
# 4) Chunking Settings
# ----------------------------
CHUNK_SIZE = 900
CHUNK_OVERLAP = 120

# Optional advanced mode: parent/child (recommended once multi-source grows)
USE_PARENT_CHILD = True
PARENT_CHUNK_SIZE = 2200
CHILD_CHUNK_SIZE = 450
CHILD_CHUNK_OVERLAP = 80

# ----------------------------
# 5) Embeddings / Retrieval Settings
# ----------------------------
EMBEDDING_MODEL_OLLAMA = "nomic-embed-text"   # example: your current Ollama embedding model
TOP_K = 8
USE_HYBRID_RETRIEVAL = True   # dense + BM25 ensemble (hybrid search)

# ----------------------------
# 6) Output Layout
# ----------------------------
TOC_DIR = OUTPUT_ROOT / "toc"
MANIFEST_PATH = OUTPUT_ROOT / "sources_manifest.json"
LOG_DIR = OUTPUT_ROOT / "logs"

for p in [OUTPUT_ROOT, TOC_DIR, LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# ----------------------------
# 7) Helpers (IDs + discovery)
# ----------------------------
def _normalize_work_id(path: Path) -> str:
    """
    Groups pdf/epub variants of the same work.
    Example: "Some Book (2019).pdf" and "Some Book (2019).epub" -> same work_id.
    """
    stem = path.stem.lower()
    stem = re.sub(r"\s+", " ", stem).strip()
    stem = re.sub(r"[\(\)\[\]\{\}]", "", stem)
    stem = re.sub(r"[^a-z0-9 _-]+", "", stem)
    return stem[:140]  # keep it short-ish

def _source_id_quick(path: Path) -> str:
    """
    Fast-ish stable ID: path + size + mtime.
    If you want stronger dedupe, switch to SHA256(file_bytes).
    """
    stat = path.stat()
    raw = f"{path.as_posix()}|{stat.st_size}|{int(stat.st_mtime)}".encode("utf-8")
    return hashlib.sha1(raw).hexdigest()  # short stable id

def discover_sources() -> List[Dict]:
    files = []
    it = SOURCES_DIR.rglob("*") if RECURSIVE else SOURCES_DIR.glob("*")
    for fp in it:
        if fp.is_file() and fp.suffix.lower() in ALLOWED_EXTS:
            files.append(fp)

    # Group by work_id (so pdf/epub duplicates are handled)
    grouped: Dict[str, List[Path]] = {}
    for fp in files:
        grouped.setdefault(_normalize_work_id(fp), []).append(fp)

    selected = []
    for work_id, variants in grouped.items():
        # choose preferred ext if multiple
        variants_sorted = sorted(
            variants,
            key=lambda p: PREFERRED_EXT_ORDER.index(p.suffix.lower())
            if p.suffix.lower() in PREFERRED_EXT_ORDER else 999
        )
        chosen = variants_sorted[0]
        selected.append({
            "work_id": work_id,
            "source_id": _source_id_quick(chosen),
            "path": str(chosen),
            "ext": chosen.suffix.lower(),
            "all_variants": [str(v) for v in variants_sorted],
        })
    return sorted(selected, key=lambda x: x["path"])

SOURCES = discover_sources()
with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
    json.dump({"unit": UNIT_CODE, "sources": SOURCES}, f, indent=2, ensure_ascii=False)

print(f"Discovered {len(SOURCES)} source works in: {SOURCES_DIR}")
print(f"Manifest written to: {MANIFEST_PATH}")


Discovered 4 source works in: inputs/sources/MBAI5004
Manifest written to: data_output/MBAI5004/sources_manifest.json


# Helper functions

In [2]:
# Cell 10: Configuration and Scoping for Content Generation (Corrected)

import os
import json
import logging

# Setup Logger for this cell
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)



# --- Global Test Overrides (for easy testing) ---
TEST_OVERRIDE_WEEKS = None
TEST_OVERRIDE_FLOW_ID = None
TEST_OVERRIDE_SESSIONS_PER_WEEK = None
TEST_OVERRIDE_DISTRIBUTION_STRATEGY = None


def process_and_load_configurations():
    """
    PHASE 1: Loads configurations, calculates a PRELIMINARY time-based slide budget,
    and saves the result as 'processed_settings.json' for the Planning Agent.
    """
    print_header("Phase 1: Configuration and Scoping Process", char="-")
    
    # --- Load all input files ---
    logger.info("Loading all necessary configuration and data files...")
    try:
        os.makedirs(CONFIG_DIR, exist_ok=True)
        with open(PARSED_UO_JSON_PATH, 'r', encoding='utf-8') as f: unit_outline = json.load(f)
        with open(PRE_EXTRACTED_TOC_JSON_PATH, 'r', encoding='utf-8') as f: book_toc = json.load(f)
        with open(SETTINGS_DECK_PATH, 'r', encoding='utf-8') as f: settings_deck = json.load(f)
        # with open(TEACHING_FLOWS_PATH, 'r', encoding='utf-8') as f: teaching_flows = json.load(f)
        logger.info("All files loaded successfully.")
    except FileNotFoundError as e:
        logger.error(f"FATAL: A required configuration file was not found: {e}")
        return None

    # --- Pre-process and Refine Settings ---
    logger.info("Pre-processing settings_deck for definitive plan...")
    processed_settings = json.loads(json.dumps(settings_deck))

    unit_info = unit_outline.get("unitInformation", {})
    processed_settings['course_id'] = unit_info.get("unitCode", "UNKNOWN_COURSE")
    processed_settings['unit_name'] = unit_info.get("unitName", "Unknown Unit Name")
    
    # --- Apply test overrides IF they are not None ---
    logger.info("Applying overrides if specified...")
    # This block now correctly sets the teaching_flow_id based on the interactive flag.
    if TEST_OVERRIDE_FLOW_ID is not None:
        processed_settings['teaching_flow_id'] = TEST_OVERRIDE_FLOW_ID
        logger.info(f"OVERRIDE: teaching_flow_id set to '{TEST_OVERRIDE_FLOW_ID}'")
    else:
        # If no override, use the 'interactive' boolean from the file as the source of truth.
        is_interactive = processed_settings.get('interactive', False)
        if is_interactive:
            processed_settings['teaching_flow_id'] = 'apply_topic_interactive'
        else:
            processed_settings['teaching_flow_id'] = 'standard_lecture'
        logger.info(f"Loaded from settings: 'interactive' is {is_interactive}. Set teaching_flow_id to '{processed_settings['teaching_flow_id']}'.")

    # The 'interactive' flag is now always consistent with the teaching_flow_id.
    processed_settings['interactive'] = "interactive" in processed_settings['teaching_flow_id'].lower()
    
    if TEST_OVERRIDE_SESSIONS_PER_WEEK is not None:
        processed_settings['week_session_setup']['sessions_per_week'] = TEST_OVERRIDE_SESSIONS_PER_WEEK
        logger.info(f"OVERRIDE: sessions_per_week set to {TEST_OVERRIDE_SESSIONS_PER_WEEK}")

    if TEST_OVERRIDE_DISTRIBUTION_STRATEGY is not None:
        processed_settings['week_session_setup']['distribution_strategy'] = TEST_OVERRIDE_DISTRIBUTION_STRATEGY
        logger.info(f"OVERRIDE: distribution_strategy set to '{TEST_OVERRIDE_DISTRIBUTION_STRATEGY}'")

    if TEST_OVERRIDE_WEEKS is not None:
        processed_settings['generation_scope']['weeks'] = TEST_OVERRIDE_WEEKS
        logger.info(f"OVERRIDE: generation_scope weeks set to {TEST_OVERRIDE_WEEKS}")

    
    
    
    # --- DYNAMIC SLIDE BUDGET CALCULATION (Phase 1) ---
    logger.info("Calculating preliminary slide budget based on session time...")
    
    params = processed_settings.get('parameters_slides', {})
    SLIDES_PER_HOUR = params.get('slides_per_hour', 18)
    
    duration_hours = processed_settings['week_session_setup'].get('session_time_duration_in_hour', 1.0)
    sessions_per_week = processed_settings['week_session_setup'].get('sessions_per_week', 1)
    
    logger.info(f"⚠️Duration Hours {duration_hours}.")
    logger.info(f"⚠️Sessions per week {sessions_per_week}.")
    
    slides_content_per_session = int(duration_hours * SLIDES_PER_HOUR)
    target_total_slides = slides_content_per_session * sessions_per_week
    
    processed_settings['slide_count_strategy']['target_total_slides'] = target_total_slides
    processed_settings['slide_count_strategy']['slides_content_per_session'] = slides_content_per_session
    logger.info(f"⚠️Preliminary weekly content slide target calculated: #️⃣{target_total_slides} slides.")
    
    # --- Resolve Generation Scope if not overridden ---
    if TEST_OVERRIDE_WEEKS is None and processed_settings.get('generation_scope', {}).get('weeks') == "all":
        num_weeks = len(unit_outline.get('weeklySchedule', []))
        processed_settings['generation_scope']['weeks'] = list(range(1, num_weeks + 1))
    
    # --- Save the processed settings to disk ---
    logger.info(f"Saving preliminary processed configuration to: {PROCESSED_SETTINGS_PATH}")
    with open(PROCESSED_SETTINGS_PATH, 'w', encoding='utf-8') as f:
        json.dump(processed_settings, f, indent=2)
    logger.info("File saved successfully.")

    # --- Assemble master config for optional preview ---
    master_config = {
        "processed_settings": processed_settings,
        "unit_outline": unit_outline,
        "book_toc": book_toc,
        # "teaching_flows": teaching_flows
    }
    
    print_header("Phase 1 Configuration Complete", char="-")
    logger.info("Master configuration object is ready for the Planning Agent.")
    return master_config



# Extract TOC from epub or PDF 

In [3]:
# Cell 4: Extract ToC for EACH source + classify (book vs publication)

from ebooklib import epub, ITEM_NAVIGATION
from bs4 import BeautifulSoup
import fitz  # PyMuPDF
import json, os, re, urllib.parse
from pathlib import Path
from typing import List, Dict, Tuple, Optional

# ----------------------------
# Patterns for doc-type signals
# ----------------------------
DOI_RE = re.compile(r"\b10\.\d{4,9}/[-._;()/:A-Z0-9]+\b", re.IGNORECASE)  # Crossref regex :contentReference[oaicite:4]{index=4}
ISBN_RE = re.compile(r"\b(?:ISBN(?:-1[03])?:?\s*)?(97[89][-\s]?\d{1,5}[-\s]?\d{1,7}[-\s]?\d{1,7}[-\s]?\d)\b", re.IGNORECASE)

def clean_epub_href(href: str) -> str:
    if not href: return ""
    cleaned_href = href.split('#')[0]
    return urllib.parse.unquote(cleaned_href)

def parse_navpoint(navpoint: BeautifulSoup, counter: List[int], level: int = 0) -> Optional[Dict]:
    title = navpoint.navLabel.text.strip()
    if not title: return None
    content_tag = navpoint.find('content', recursive=False)
    link_filename = clean_epub_href(content_tag['src']) if content_tag else ""
    node = {"level": level, "toc_id": counter[0], "title": title, "link_filename": link_filename, "children": []}
    counter[0] += 1
    for child in navpoint.find_all('navPoint', recursive=False):
        child_node = parse_navpoint(child, counter, level + 1)
        if child_node: node["children"].append(child_node)
    return node

def parse_li(li_element: BeautifulSoup, counter: List[int], level: int = 0) -> Optional[Dict]:
    a_tag = li_element.find('a', recursive=False)
    if not a_tag: return None
    title = a_tag.get_text(strip=True)
    if not title: return None
    link_filename = clean_epub_href(a_tag.get('href'))
    node = {"level": level, "toc_id": counter[0], "title": title, "link_filename": link_filename, "children": []}
    counter[0] += 1
    nested_ol = li_element.find('ol', recursive=False)
    if nested_ol:
        for sub_li in nested_ol.find_all('li', recursive=False):
            child_node = parse_li(sub_li, counter, level + 1)
            if child_node: node["children"].append(child_node)
    return node

def extract_epub_toc_and_meta(epub_path: str) -> Tuple[List[Dict], Dict]:
    book = epub.read_epub(epub_path)
    toc_data: List[Dict] = []
    id_counter = [0]

    # EPUB metadata (DC)
    meta = {
        "title": (book.get_metadata("DC", "title") or [("", {})])[0][0],
        "creator": (book.get_metadata("DC", "creator") or [("", {})])[0][0],
        "identifier": (book.get_metadata("DC", "identifier") or [("", {})])[0][0],
        "language": (book.get_metadata("DC", "language") or [("", {})])[0][0],
        "publisher": (book.get_metadata("DC", "publisher") or [("", {})])[0][0],
    }

    for nav_item in book.get_items_of_type(ITEM_NAVIGATION):
        soup = BeautifulSoup(nav_item.get_content(), 'xml')
        if nav_item.get_name().endswith('.ncx'):
            navmap = soup.find('navMap')
            if navmap:
                for navpoint in navmap.find_all('navPoint', recursive=False):
                    node = parse_navpoint(navpoint, id_counter, level=0)
                    if node: toc_data.append(node)
        else:
            toc_nav = soup.select_one('nav[epub|type="toc"]')
            if toc_nav:
                top_ol = toc_nav.find('ol', recursive=False)
                if top_ol:
                    for li in top_ol.find_all('li', recursive=False):
                        node = parse_li(li, id_counter, level=0)
                        if node: toc_data.append(node)
        if toc_data:
            break

    return toc_data, meta

def build_pdf_hierarchy_with_ids(toc_list: List) -> List[Dict]:
    root = []
    parent_stack = {-1: {"children": root}}
    id_counter = [0]
    for level, title, page, *_ in toc_list:
        normalized_level = level - 1
        node = {
            "level": normalized_level,
            "toc_id": id_counter[0],
            "title": (title or "").strip(),
            "page": page,          # 1-based per PyMuPDF docs :contentReference[oaicite:5]{index=5}
            "children": []
        }
        id_counter[0] += 1
        parent = parent_stack.get(normalized_level - 1)
        if parent:
            parent["children"].append(node)
        parent_stack[normalized_level] = node
    return root

def pdf_first_pages_text(doc: fitz.Document, n_pages: int = 2) -> str:
    parts = []
    for i in range(min(n_pages, doc.page_count)):
        parts.append(doc.load_page(i).get_text("text") or "")
    return "\n".join(parts)

def flatten_toc_count(toc_nodes: List[Dict]) -> int:
    if not toc_nodes: return 0
    count = 0
    stack = list(toc_nodes)
    while stack:
        node = stack.pop()
        count += 1
        stack.extend(node.get("children", []))
    return count

def classify_doc_kind(ext: str, page_count: int, toc_nodes: List[Dict], meta: Dict, sample_text: str) -> Tuple[str, Dict]:
    signals = {
        "doi": None,
        "isbn": None,
        "has_toc": bool(toc_nodes),
        "toc_entries": flatten_toc_count(toc_nodes),
        "page_count": page_count,
    }

    doi_match = DOI_RE.search(sample_text or "")
    if doi_match:
        signals["doi"] = doi_match.group(0)

    isbn_match = ISBN_RE.search((sample_text or "") + " " + (meta.get("identifier") or ""))
    if isbn_match:
        signals["isbn"] = isbn_match.group(1)

    text_lower = (sample_text or "").lower()
    has_abstract = "abstract" in text_lower
    has_references = "references" in text_lower or "bibliography" in text_lower

    score_pub = 0
    score_book = 0

    if signals["doi"]: score_pub += 3
    if has_abstract: score_pub += 1
    if has_references: score_pub += 1
    if page_count and page_count < 60: score_pub += 1

    if signals["isbn"]: score_book += 3
    if signals["toc_entries"] >= 30: score_book += 2
    if page_count and page_count >= 120: score_book += 2

    doc_kind = "publication" if score_pub > score_book else "book"
    return doc_kind, signals

def extract_pdf_toc_and_meta(pdf_path: str) -> Tuple[List[Dict], Dict, int, str]:
    doc = fitz.open(pdf_path)
    toc_raw = doc.get_toc()  # outlines/bookmarks :contentReference[oaicite:6]{index=6}
    toc_nodes = build_pdf_hierarchy_with_ids(toc_raw) if toc_raw else []
    meta = doc.metadata or {}               # PyMuPDF metadata :contentReference[oaicite:7]{index=7}
    page_count = doc.page_count             # page_count :contentReference[oaicite:8]{index=8}
    sample_text = pdf_first_pages_text(doc, n_pages=2)
    doc.close()
    return toc_nodes, meta, page_count, sample_text

# ----------------------------
# MAIN: loop over SOURCES
# ----------------------------
TOC_DIR = Path(TOC_DIR)  # ensure Path
TOC_DIR.mkdir(parents=True, exist_ok=True)

enriched_sources = []
for s in SOURCES:
    path = s["path"]
    ext = s["ext"]
    source_id = s["source_id"]

    toc_out = TOC_DIR / f"{source_id}_toc.json"
    meta_out = TOC_DIR / f"{source_id}_meta.json"

    toc_nodes: List[Dict] = []
    meta: Dict = {}
    page_count = 0
    sample_text = ""

    try:
        if ext == ".epub":
            toc_nodes, meta = extract_epub_toc_and_meta(path)   # EbookLib metadata :contentReference[oaicite:9]{index=9}
        elif ext == ".pdf":
            toc_nodes, meta, page_count, sample_text = extract_pdf_toc_and_meta(path)
        else:
            raise ValueError(f"Unsupported ext: {ext}")

        # Classification (book vs publication)
        doc_kind, signals = classify_doc_kind(ext, page_count, toc_nodes, meta, sample_text)

        # Write per-source TOC + meta
        with open(toc_out, "w", encoding="utf-8") as f:
            json.dump(toc_nodes, f, indent=2, ensure_ascii=False)

        with open(meta_out, "w", encoding="utf-8") as f:
            json.dump({"meta": meta, "signals": signals, "doc_kind": doc_kind}, f, indent=2, ensure_ascii=False)

        enriched = dict(s)
        enriched.update({
            "toc_json": str(toc_out),
            "meta_json": str(meta_out),
            "doc_kind": doc_kind,
            **signals
        })
        enriched_sources.append(enriched)

        print(f"✅ {doc_kind.upper():11} | {ext:5} | toc={signals['toc_entries']:4} | pages={page_count:4} | {path}")

    except Exception as e:
        print(f"❌ Failed: {path} :: {e}")
        enriched = dict(s)
        enriched.update({"error": str(e)})
        enriched_sources.append(enriched)

# Update manifest (so downstream chunking/retrieval knows doc_kind + toc path)
with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
    json.dump({"sources": enriched_sources}, f, indent=2, ensure_ascii=False)

print(f"\nManifest updated: {MANIFEST_PATH}")
print(f"ToCs written under: {TOC_DIR}")


✅ PUBLICATION | .pdf  | toc=   0 | pages=  14 | inputs/sources/MBAI5004/Ethical AI in Information Technology Navigating Bias, Privacy, Transparency,.pdf
✅ BOOK        | .pdf  | toc=  30 | pages= 533 | inputs/sources/MBAI5004/Ethics Of Artificial Intelligence -- S_ Matthew Liao.pdf
✅ BOOK        | .pdf  | toc=  90 | pages= 121 | inputs/sources/MBAI5004/Ethics of Artificial Intelligence Case Studies.pdf
✅ BOOK        | .pdf  | toc= 260 | pages= 392 | inputs/sources/MBAI5004/Ethics, Governance, and Policies in Artificial Intelligence -- Luciano Floridi.pdf

Manifest updated: data_output/MBAI5004/sources_manifest.json
ToCs written under: data_output/MBAI5004/toc


# Hirachical DB base on TOC

https://chatgpt.com/c/WEB:dbf26b2e-f629-46d7-8380-ba6c2a501898 

## Process Books

In [4]:
# ===========================⭐⭐⭐⭐=================================
# CELL X — MULTI-SOURCE PDF PRE-EXTRACTION
#   (Anchors + Parent Intro + AUTO Subsections + NO-TOC STRUCTURING)
#
# Writes per-source folder (by source_id):
#   - toc.json  (includes augmented_flat_entries + augmented_tree)
#   - sections.jsonl
#   - sections_text/<tree...>/(intro.txt | section.txt | full_with_children.txt)
#
# Key fixes:
#   ✅ Always returns "flip_y_used" in ALL branches (no KeyError in print)
#   ✅ Optional: if PDF has NO ToC, build synthetic section tree from numbered headings
#
# NEW fixes:
#   ✅ Two-column PDFs: column-aware reading order (left column then right column)
#   ✅ Stop "no-ToC structuring" at REFERENCES (avoid treating refs 1.,2.,3... as headings)
#   ✅ Extra heuristic to skip reference-like numbered lines (year/doi) even if cutoff missed
# ============================================================

from __future__ import annotations
import json
import re
import hashlib
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
from difflib import SequenceMatcher

import fitz  # PyMuPDF


# -----------------------------
# Uses your globals from Cell 1
# -----------------------------
MANIFEST_PATH = Path(MANIFEST_PATH)
OUTPUT_ROOT = Path(OUTPUT_ROOT)

PRE_RAG_DIR = OUTPUT_ROOT / "pre_rag_pdf_sections"
PRE_RAG_DIR.mkdir(parents=True, exist_ok=True)


# -----------------------------
# Extraction options
# -----------------------------
SORT_BLOCKS = True
DROP_HEADERS_FOOTERS = True
HEADER_PX = 50.0
FOOTER_PX = 55.0

TITLE_MATCH_RATIO = 0.74
TITLE_SNAP_Y_WINDOW_UP = 120.0
TITLE_SNAP_Y_WINDOW_DOWN = 380.0

ANCHOR_Y_PROX = 90.0
SCORE_SAMPLE_N = 30

MAX_PREVIEW_CHARS = 2500

# -----------------------------
# Auto subsection discovery (within a ToC parent range)
# -----------------------------
DISCOVER_NUMBERED_SUBSECTIONS = True
DISCOVER_ONLY_IF_NO_TOC_CHILDREN = True   # conservative default (recommended)
MAX_HEADING_LINE_LEN = 140

# 1.1. Introduction   OR   1.1 Introduction   OR   1.2.3 Some title
SUBHEADING_RE = re.compile(r"^\s*(?P<num>\d+(?:\.\d+)+)\.?\s+(?P<title>.+?)\s*$")

# top-level: 1 Introduction OR 2. Methods
TOPHEADING_RE = re.compile(r"^\s*(?P<num>\d+)\.?\s+(?P<title>.+?)\s*$")

# avoid capturing Fig. 2.1, Table 3.2, etc.
BAD_HEADING_PREFIXES = ("fig", "figure", "table", "eq", "equation")

# -----------------------------
# NEW: No-ToC structuring from numbered headings
# -----------------------------
STRUCTURE_WHEN_NO_TOC = True
NO_TOC_MIN_HEADINGS = 5          # if fewer than this, fallback to per-page export
NO_TOC_MAX_HEADINGS = 600        # safety cap
NO_TOC_MAX_DEPTH = 5             # max dots depth (e.g., 1.2.3.4.5)

# -----------------------------
# NEW: Two-column handling + References cut-off
# -----------------------------
ENABLE_COLUMN_AWARE_ORDER = True

# Column split detector (heuristics)
COLUMN_DETECT_MIN_BLOCKS = 10
COLUMN_SPLIT_SEARCH_MIN_FRAC = 0.35
COLUMN_SPLIT_SEARCH_MAX_FRAC = 0.65
COLUMN_SPLIT_STEPS = 25
COLUMN_MAX_CROSS_FRAC = 0.08       # allow up to 8% blocks crossing the split
COLUMN_MIN_SIDE_FRAC = 0.20        # each side must have >= 20% of blocks

# References cutoff (stop scanning headings after references)
STOP_AT_REFERENCES = True
REF_CUTOFF_SCAN_LAST_PAGES = 10
REFERENCES_HEADING_RE = re.compile(r"^\s*(references|bibliography|works\s+cited)\s*$", re.IGNORECASE)

# Extra safety: skip "reference-like" numbered lines even if cutoff missed
SKIP_REFERENCE_LIKE_HEADINGS = True


# ============================================================
# Helpers
# ============================================================

def _sha1(s: str) -> str:
    return hashlib.sha1(s.encode("utf-8", errors="ignore")).hexdigest()

def _norm(s: str) -> str:
    s = (s or "").lower()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"[^\w\s]", "", s)
    return s.strip()

def _slug(s: str, max_len: int = 42) -> str:
    s = (s or "")
    s = s.replace("’", "'")
    s = re.sub(r"\s+", "_", s.strip())
    s = re.sub(r"[^A-Za-z0-9_\-]+", "", s)
    return (s or "untitled")[:max_len]

def _safe_seg(toc_id: int, title: str) -> str:
    h = _sha1(title)[:6]
    return f"toc{toc_id:04d}__{_slug(title)}__{h}"

def _similar(a: str, b: str) -> float:
    return SequenceMatcher(None, _norm(a), _norm(b)).ratio()

# Caches keyed by (doc_id, page_no)
_COLUMN_SPLIT_CACHE: Dict[Tuple[int, int], Optional[float]] = {}
_REFERENCES_CUTOFF_CACHE: Dict[int, Optional[Tuple[int, int, float]]] = {}


def _looks_like_reference_entry(title_part: str) -> bool:
    """
    Heuristic to avoid treating reference list items as headings.
    Trigger on patterns commonly found in references: (2020), doi, 'et al.', many commas.
    """
    t = (title_part or "").strip()
    if not t:
        return False
    if re.search(r"\(\s*\d{4}[a-z]?\s*\)", t):
        return True
    if re.search(r"\bdoi\b", t, flags=re.IGNORECASE):
        return True
    if re.search(r"\bet\s+al\.?\b", t, flags=re.IGNORECASE):
        return True
    # lots of commas is common in author lists
    if t.count(",") >= 3 and len(t) > 35:
        return True
    if re.search(r"https?://", t, flags=re.IGNORECASE):
        return True
    return False


def _detect_column_split_from_blocks(blocks: List[Tuple], page_width: float) -> Optional[float]:
    """
    Returns split_x if the page looks like a 2-column layout, else None.
    We choose a split that minimizes "crossing blocks" and has reasonable balance.
    """
    if not ENABLE_COLUMN_AWARE_ORDER:
        return None
    if len(blocks) < COLUMN_DETECT_MIN_BLOCKS:
        return None
    if page_width <= 0:
        return None

    # Candidate splits
    x_min = page_width * COLUMN_SPLIT_SEARCH_MIN_FRAC
    x_max = page_width * COLUMN_SPLIT_SEARCH_MAX_FRAC
    if x_max <= x_min:
        return None
    if COLUMN_SPLIT_STEPS < 5:
        cands = [0.5 * page_width]
    else:
        step = (x_max - x_min) / float(COLUMN_SPLIT_STEPS)
        cands = [x_min + i * step for i in range(COLUMN_SPLIT_STEPS + 1)]

    best_s = None
    best_score = None

    for s in cands:
        cross = 0
        left = 0
        right = 0
        for b in blocks:
            if not b or len(b) < 5:
                continue
            x0, y0, x1, y1 = float(b[0]), float(b[1]), float(b[2]), float(b[3])
            if x0 < s < x1:
                cross += 1
            elif x1 <= s:
                left += 1
            elif x0 >= s:
                right += 1

        # Require both sides to exist
        if left == 0 or right == 0:
            continue

        # Penalize crossings heavily; secondary: balance
        score = (cross * 10) + abs(left - right)

        if best_score is None or score < best_score:
            best_score = score
            best_s = s

    if best_s is None:
        return None

    # Validate best split
    cross = 0
    left = 0
    right = 0
    for b in blocks:
        if not b or len(b) < 5:
            continue
        x0, x1 = float(b[0]), float(b[2])
        if x0 < best_s < x1:
            cross += 1
        elif x1 <= best_s:
            left += 1
        elif x0 >= best_s:
            right += 1

    n = max(1, left + right + cross)
    if cross > max(2, int(n * COLUMN_MAX_CROSS_FRAC)):
        return None
    if left < int(n * COLUMN_MIN_SIDE_FRAC) or right < int(n * COLUMN_MIN_SIDE_FRAC):
        return None

    return float(best_s)


def _get_page_split_x(page: fitz.Page) -> Optional[float]:
    doc_id = id(page.parent)
    key = (doc_id, int(page.number))
    if key in _COLUMN_SPLIT_CACHE:
        return _COLUMN_SPLIT_CACHE[key]

    # Use raw blocks (unsorted) for detection
    raw = page.get_text("blocks", sort=False)
    raw = [b for b in raw if b and len(b) >= 5]
    split_x = _detect_column_split_from_blocks(raw, float(page.rect.width))
    _COLUMN_SPLIT_CACHE[key] = split_x
    return split_x


def _block_list(page: fitz.Page) -> List[Tuple]:
    """
    Returns blocks filtered (optional headers/footers) and ordered in reading order.

    - For single-column: y then x
    - For two-column: left column (col=0) then right column (col=1), each y then x

    NOTE: We append the computed column id as the LAST element of the tuple.
    """
    # get blocks without relying on PyMuPDF's sort (we'll order ourselves)
    blocks = page.get_text("blocks", sort=False)
    blocks = [b for b in blocks if b and len(b) >= 5]

    if DROP_HEADERS_FOOTERS:
        h = float(page.rect.height)
        out = []
        for b in blocks:
            y0, y1 = float(b[1]), float(b[3])
            if y1 < HEADER_PX:
                continue
            if y0 > (h - FOOTER_PX):
                continue
            out.append(b)
        blocks = out

    # Column-aware ordering
    split_x = _get_page_split_x(page) if ENABLE_COLUMN_AWARE_ORDER else None

    enriched: List[Tuple] = []
    if split_x is not None:
        for b in blocks:
            x0, y0, x1, y1, text, *rest = b
            x0f, x1f = float(x0), float(x1)
            if x0f >= split_x:
                col = 1
            elif x1f <= split_x:
                col = 0
            else:
                col = 0  # spanning block: treat as left
            enriched.append((x0, y0, x1, y1, text, *rest, col))
        blocks = sorted(enriched, key=lambda t: (int(t[-1]), float(t[1]), float(t[0])))
    else:
        # Single column: sort by y then x if requested
        if SORT_BLOCKS:
            blocks = sorted(blocks, key=lambda t: (float(t[1]), float(t[0])))
        # append col=0 for consistency
        blocks = [(b[0], b[1], b[2], b[3], b[4], *b[5:], 0) for b in blocks]

    return blocks


def _find_heading_block_y(page: fitz.Page, y_hint: float, title: str) -> Optional[float]:
    blocks = _block_list(page)
    y0_min = max(0.0, y_hint - TITLE_SNAP_Y_WINDOW_UP)
    y0_max = min(float(page.rect.height), y_hint + TITLE_SNAP_Y_WINDOW_DOWN)

    best = None
    best_score = 0.0
    for (x0, y0, x1, y1, text, *rest) in blocks:
        y0f = float(y0)
        if y0f < y0_min or y0f > y0_max:
            continue
        t = (text or "").strip()
        if not t:
            continue
        score = max(_similar(t, title), 1.0 if _norm(title) in _norm(t) else 0.0)
        if score > best_score:
            best_score = score
            best = y0f

    if best is not None and best_score >= TITLE_MATCH_RATIO:
        return best
    return None


def _extract_blocks_between(page: fitz.Page, y_start: float, y_end: Optional[float]) -> str:
    """
    Y-based extraction (single-column friendly). For two-column pages, use reading-order
    extraction via _extract_between_anchors (which uses col-aware boundaries).
    """
    h = float(page.rect.height)
    y_start = max(0.0, min(y_start, h))
    if y_end is not None:
        y_end = max(0.0, min(y_end, h))
        if y_end <= y_start:
            return ""

    blocks = _block_list(page)
    out: List[str] = []
    for (x0, y0, x1, y1, text, *rest) in blocks:
        y0f, y1f = float(y0), float(y1)
        if y1f <= y_start:
            continue
        if y_end is not None and y0f >= y_end:
            continue
        t = (text or "").strip()
        if t:
            out.append(t)

    joined = "\n".join(out)
    joined = re.sub(r"\n{3,}", "\n\n", joined).strip()
    return joined


def _extract_range(
    doc: fitz.Document,
    start_page0: int,
    start_y: float,
    end_page0: Optional[int],
    end_y: Optional[float],
) -> str:
    """
    Legacy y-based range extraction (kept for ToC/book mode).
    """
    n_pages = doc.page_count
    start_page0 = max(0, min(start_page0, n_pages - 1))

    if end_page0 is None:
        end_page0 = n_pages - 1
        end_y = None
    else:
        end_page0 = max(0, min(end_page0, n_pages - 1))

    if end_page0 < start_page0:
        end_page0 = start_page0
        end_y = None

    parts: List[str] = []
    if start_page0 == end_page0:
        parts.append(_extract_blocks_between(doc.load_page(start_page0), start_y, end_y))
    else:
        parts.append(_extract_blocks_between(doc.load_page(start_page0), start_y, None))
        for p in range(start_page0 + 1, end_page0):
            parts.append(_extract_blocks_between(doc.load_page(p), 0.0, None))
        parts.append(_extract_blocks_between(doc.load_page(end_page0), 0.0, end_y))

    text = "\n\n".join([p for p in parts if p.strip()]).strip()
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    return text


def _extract_range_reading_order(
    doc: fitz.Document,
    start_anchor: Tuple[int, float, int],
    end_anchor: Optional[Tuple[int, float, int]],
) -> str:
    """
    Column-aware reading-order extraction.
    Anchors are (page0, y, col) where col=0(left) or 1(right).

    Reading order: page asc, col asc, y asc, x asc.
    """
    s_p0, s_y, s_col = int(start_anchor[0]), float(start_anchor[1]), int(start_anchor[2])

    if end_anchor is None:
        e_p0, e_y, e_col = (doc.page_count - 1, None, 99)
    else:
        e_p0, e_y, e_col = int(end_anchor[0]), float(end_anchor[1]), int(end_anchor[2])

    if e_p0 < s_p0:
        e_p0, e_y, e_col = (s_p0, None, 99)
    if (s_p0 == e_p0) and (e_y is not None) and (e_col == s_col) and (e_y <= s_y):
        return ""

    parts: List[str] = []

    for p0 in range(s_p0, e_p0 + 1):
        page = doc.load_page(p0)
        blocks = _block_list(page)  # includes col as last element and ordered

        for (x0, y0, x1, y1, text, *rest) in blocks:
            col = int(rest[-1]) if rest else 0
            y0f, y1f = float(y0), float(y1)

            # start bound
            if p0 == s_p0:
                if col < s_col:
                    continue
                if col == s_col and y1f <= s_y:
                    continue

            # end bound
            if end_anchor is not None and p0 == e_p0:
                if col > e_col:
                    break
                if col == e_col and e_y is not None and y0f >= e_y:
                    break

            t = (text or "").strip()
            if t:
                parts.append(t)

    joined = "\n".join(parts)
    joined = re.sub(r"\n{3,}", "\n\n", joined).strip()
    return joined


def _extract_between_anchors(
    doc: fitz.Document,
    start_anchor: Tuple,
    end_anchor: Optional[Tuple],
) -> str:
    """
    Wrapper: if anchors include column info (len>=3), use reading-order extraction.
    Otherwise, fall back to legacy y-based extraction.
    """
    if len(start_anchor) >= 3 and (end_anchor is None or len(end_anchor) >= 3):
        return _extract_range_reading_order(doc, start_anchor, end_anchor)

    end_p0 = None if end_anchor is None else int(end_anchor[0])
    end_y = None if end_anchor is None else float(end_anchor[1])
    return _extract_range(doc, int(start_anchor[0]), float(start_anchor[1]), end_p0, end_y)


def _boundary_check(section_text: str, this_title: str, next_title: Optional[str]) -> Dict[str, Any]:
    head = (section_text or "")[:1800]
    ok_start = (_norm(this_title) in _norm(head)) or (_similar(head, this_title) >= 0.35)
    ok_end = True
    if next_title:
        ok_end = (_norm(next_title) not in _norm(head))
    return {"ok_start": bool(ok_start), "ok_end": bool(ok_end)}


def _extract_page_text_column_aware(page: fitz.Page) -> str:
    """
    Better per-page fallback for multi-column: join ordered blocks.
    """
    blocks = _block_list(page)
    out: List[str] = []
    for (x0, y0, x1, y1, text, *rest) in blocks:
        t = (text or "").strip()
        if t:
            out.append(t)
    joined = "\n".join(out)
    joined = re.sub(r"\n{3,}", "\n\n", joined).strip()
    return joined


# ============================================================
# ToC extraction + coordinate orientation selection
# ============================================================

def _get_detailed_toc(doc: fitz.Document) -> List[List[Any]]:
    return doc.get_toc(simple=False)

def _to_y_float(to: Any) -> float:
    # dest["to"] can be fitz.Point OR tuple/list OR something else
    if to is None:
        return 0.0
    if hasattr(to, "y"):
        return float(getattr(to, "y", 0.0))
    if isinstance(to, (list, tuple)) and len(to) >= 2:
        try:
            return float(to[1])
        except Exception:
            return 0.0
    return 0.0

def _score_orientation(doc: fitz.Document, toc_items: List[List[Any]], flip_y: bool) -> int:
    score = 0
    cache_blocks: Dict[int, List[Tuple]] = {}

    for row in toc_items[:SCORE_SAMPLE_N]:
        if len(row) < 4:
            continue
        title, page1, dest = row[1], row[2], row[3]
        if not isinstance(dest, dict):
            continue
        p0 = int(dest.get("page", (page1 - 1 if isinstance(page1, int) else 0)))
        if p0 < 0 or p0 >= doc.page_count:
            continue

        to = dest.get("to", None)
        if to is None:
            continue

        page = doc.load_page(p0)
        h = float(page.rect.height)
        y = _to_y_float(to)
        y = (h - y) if flip_y else y

        if p0 not in cache_blocks:
            cache_blocks[p0] = _block_list(page)

        title_s = (title or "").strip()
        if not title_s:
            continue

        found = False
        for (x0, y0, x1, y1, text, *rest) in cache_blocks[p0]:
            if abs(float(y0) - y) > ANCHOR_Y_PROX:
                continue
            t = (text or "").strip()
            if not t:
                continue
            if _norm(title_s) in _norm(t) or _similar(t, title_s) >= TITLE_MATCH_RATIO:
                found = True
                break

        if found:
            score += 1

    return score

def _build_tree(entries: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    by_id = {e["toc_id"]: {k: e[k] for k in e.keys() if k not in ("parent_toc_id",)} for e in entries}
    root: List[Dict[str, Any]] = []
    for e in entries:
        node = by_id[e["toc_id"]]
        node["children"] = []
        pid = e.get("parent_toc_id")
        if pid is None:
            root.append(node)
        else:
            parent = by_id.get(pid)
            if parent is not None:
                parent["children"].append(node)
    return root

def _anchor_of_entry(e: Dict[str, Any]) -> Tuple:
    # For structured-from-text headings we carry "col" (0/1) to enable reading-order extraction.
    page0 = int(e["page_0based"])
    y = float(e.get("y_snapped", e.get("y", 0.0)))
    if "col" in e and isinstance(e["col"], int):
        return (page0, y, int(e["col"]))
    return (page0, y)

def _compute_end_anchors(entries: List[Dict[str, Any]], doc: fitz.Document) -> None:
    n = len(entries)
    for i in range(n):
        cur = entries[i]
        cur_lvl = int(cur["level"])

        end_full: Optional[Tuple] = None
        next_peer_title: Optional[str] = None
        for j in range(i + 1, n):
            nxt = entries[j]
            if int(nxt["level"]) <= cur_lvl:
                end_full = _anchor_of_entry(nxt)
                next_peer_title = nxt["title"]
                break

        cur["end_full"] = end_full
        cur["next_peer_title"] = next_peer_title or ""

        kids = cur.get("children_toc_ids", [])
        if kids:
            first_child = next((x for x in entries if x["toc_id"] == kids[0]), None)
            cur["end_intro"] = _anchor_of_entry(first_child) if first_child else end_full
        else:
            cur["end_intro"] = end_full

def _build_entries_with_paths_and_anchors(doc: fitz.Document) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]], bool]:
    toc_items = _get_detailed_toc(doc)
    if not toc_items:
        return [], [], False

    raw_score = _score_orientation(doc, toc_items, flip_y=False)
    flip_score = _score_orientation(doc, toc_items, flip_y=True)
    flip_y = (flip_score > raw_score)

    entries: List[Dict[str, Any]] = []
    stack_titles: List[str] = []
    stack_ids: List[int] = []

    toc_id = 0
    for row in toc_items:
        if len(row) < 3:
            continue
        lvl, title, page1 = row[0], row[1], row[2]
        dest = row[3] if len(row) >= 4 else None

        if not isinstance(lvl, int) or lvl < 1:
            continue
        title = (title or "").strip()
        if not title:
            continue
        if not isinstance(page1, int) or page1 < 1 or page1 > doc.page_count:
            continue

        if isinstance(dest, dict) and "page" in dest:
            page0 = int(dest["page"])
        else:
            page0 = page1 - 1
        page0 = max(0, min(page0, doc.page_count - 1))

        page = doc.load_page(page0)
        h = float(page.rect.height)

        y = 0.0
        if isinstance(dest, dict) and dest.get("to", None) is not None:
            y = _to_y_float(dest["to"])
            if flip_y:
                y = (h - y)

        level0 = lvl - 1

        stack_titles = stack_titles[:level0]
        stack_ids = stack_ids[:level0]
        parent_toc_id = stack_ids[-1] if stack_ids else None

        stack_titles.append(title)
        stack_ids.append(toc_id)

        e = {
            "toc_id": toc_id,
            "level": level0,
            "title": title,
            "titles_path": stack_titles.copy(),
            "toc_path_ids": stack_ids.copy(),
            "parent_toc_id": parent_toc_id,
            "page_0based": page0,
            "page_1based": page0 + 1,
            "y": float(y),
            "synthetic": False,
        }
        entries.append(e)
        toc_id += 1

    entries.sort(key=lambda e: (e["page_0based"], e["y"], e["toc_id"]))

    by_id = {e["toc_id"]: e for e in entries}
    children_map: Dict[int, List[int]] = {}
    for e in entries:
        pid = e.get("parent_toc_id")
        if pid is not None:
            children_map.setdefault(int(pid), []).append(int(e["toc_id"]))
    for pid, kids in children_map.items():
        kids.sort()
        by_id[pid]["children_toc_ids"] = kids
        by_id[pid]["first_child_toc_id"] = kids[0]

    # snap y to nearest real heading block y
    for e in entries:
        p0 = e["page_0based"]
        page = doc.load_page(p0)
        snapped = _find_heading_block_y(page, e["y"], e["title"])
        e["y_snapped"] = float(snapped) if snapped is not None else float(e["y"])

    return entries, _build_tree(entries), flip_y


# ============================================================
# Numbered subheadings discovery inside a range (dict lines+bbox)
# ============================================================

def _iter_lines_with_bbox(page: fitz.Page) -> List[Tuple[float, float, float, float, str, int]]:
    """
    Returns lines (x0,y0,x1,y1,text,col) in reading order.
    """
    d = page.get_text("dict")
    lines_out: List[Tuple[float, float, float, float, str, int]] = []
    h = float(page.rect.height)

    split_x = _get_page_split_x(page) if ENABLE_COLUMN_AWARE_ORDER else None

    for b in d.get("blocks", []):
        if b.get("type", 0) != 0:
            continue
        for ln in b.get("lines", []):
            bbox = ln.get("bbox", None)
            if not bbox or len(bbox) != 4:
                continue
            x0, y0, x1, y1 = map(float, bbox)

            if DROP_HEADERS_FOOTERS:
                if y1 < HEADER_PX:
                    continue
                if y0 > (h - FOOTER_PX):
                    continue

            spans = ln.get("spans", [])
            txt = "".join([(sp.get("text") or "") for sp in spans]).strip()
            txt = re.sub(r"\s+", " ", txt).strip()
            if not txt:
                continue

            if split_x is not None and x0 >= split_x:
                col = 1
            else:
                col = 0

            lines_out.append((x0, y0, x1, y1, txt, col))

    # reading order
    lines_out.sort(key=lambda t: (int(t[5]), float(t[1]), float(t[0])))
    return lines_out


def _extract_number_prefix_from_title(title: str) -> Optional[str]:
    """
    If title starts with a number prefix like:
      "1 Ethical Learning" -> "1"
      "1.2 A Social Perspective" -> "1.2"
      "2.3.1 Something" -> "2.3.1"
    """
    m = re.match(r"^\s*(\d+(?:\.\d+)*)\b", title or "")
    return m.group(1) if m else None


def _find_references_cutoff(doc: fitz.Document) -> Optional[Tuple[int, int, float]]:
    """
    Find the anchor (page0, col, y0) where the References section begins.
    We scan only the last REF_CUTOFF_SCAN_LAST_PAGES pages for speed.
    """
    doc_id = id(doc)
    if doc_id in _REFERENCES_CUTOFF_CACHE:
        return _REFERENCES_CUTOFF_CACHE[doc_id]

    if not STOP_AT_REFERENCES or doc.page_count <= 0:
        _REFERENCES_CUTOFF_CACHE[doc_id] = None
        return None

    start = max(0, doc.page_count - REF_CUTOFF_SCAN_LAST_PAGES)
    cutoff: Optional[Tuple[int, int, float]] = None

    for p0 in range(start, doc.page_count):
        page = doc.load_page(p0)
        for (x0, y0, x1, y1, line, col) in _iter_lines_with_bbox(page):
            s = (line or "").strip()
            if REFERENCES_HEADING_RE.match(s):
                cutoff = (p0, int(col), float(y0))
                _REFERENCES_CUTOFF_CACHE[doc_id] = cutoff
                return cutoff

    _REFERENCES_CUTOFF_CACHE[doc_id] = None
    return None


def _find_numbered_subheadings_in_range(
    doc: fitz.Document,
    start_anchor: Tuple,
    end_anchor: Optional[Tuple],
    parent_prefix: Optional[str],
) -> List[Dict[str, Any]]:
    s_p0, s_y = int(start_anchor[0]), float(start_anchor[1])
    if end_anchor is None:
        e_p0, e_y = (doc.page_count - 1, None)
    else:
        e_p0, e_y = (int(end_anchor[0]), float(end_anchor[1]))

    refs_cutoff = _find_references_cutoff(doc)

    cands: List[Dict[str, Any]] = []
    seen = set()

    for p0 in range(s_p0, e_p0 + 1):
        # Stop entirely after references
        if refs_cutoff and p0 > refs_cutoff[0]:
            break

        page = doc.load_page(p0)
        for (x0, y0, x1, y1, line, col) in _iter_lines_with_bbox(page):
            # references cutoff on the same page (reading order)
            if refs_cutoff and p0 == refs_cutoff[0]:
                ref_p0, ref_col, ref_y0 = refs_cutoff
                if col > ref_col:
                    break
                if col == ref_col and y0 >= ref_y0:
                    break

            if p0 == s_p0 and y1 <= s_y:
                continue
            if end_anchor is not None and p0 == e_p0 and e_y is not None and y0 >= e_y:
                continue

            if len(line) > MAX_HEADING_LINE_LEN:
                continue

            low = line.lower().strip()
            if any(low.startswith(pref) for pref in BAD_HEADING_PREFIXES):
                continue

            m = SUBHEADING_RE.match(line)
            if not m:
                continue

            num = m.group("num").strip()
            ttl = m.group("title").strip()

            # prefix filter:
            #   parent "1"   -> accept "1.x"
            #   parent "1.1" -> accept "1.1.x"
            if parent_prefix:
                must = parent_prefix + "."
                if not num.startswith(must):
                    continue

            key = f"{p0}|{col}|{round(y0,1)}|{num}|{_norm(ttl)}"
            if key in seen:
                continue
            seen.add(key)

            cands.append({
                "num": num,
                "title": ttl,
                "page_0based": p0,
                "page_1based": p0 + 1,
                "y": float(y0),
                "x0": float(x0),
                "col": int(col),
            })

    cands.sort(key=lambda x: (x["page_0based"], x.get("col", 0), x["y"], x.get("x0", 0.0)))
    return cands


# ============================================================
# NEW: Build synthetic entries from numbered headings when no ToC
# ============================================================

def _scan_numbered_headings_whole_doc(doc: fitz.Document) -> List[Dict[str, Any]]:
    cands: List[Dict[str, Any]] = []
    seen = set()

    refs_cutoff = _find_references_cutoff(doc)

    for p0 in range(doc.page_count):
        # Stop entirely after references
        if refs_cutoff and p0 > refs_cutoff[0]:
            break

        page = doc.load_page(p0)
        for (x0, y0, x1, y1, line, col) in _iter_lines_with_bbox(page):
            # references cutoff on the same page (reading order)
            if refs_cutoff and p0 == refs_cutoff[0]:
                ref_p0, ref_col, ref_y0 = refs_cutoff
                if col > ref_col:
                    # once we reached columns beyond the references column, stop all
                    return sorted(cands, key=lambda x: (x["page_0based"], x.get("col", 0), x["y"], x.get("x0", 0.0)))
                if col == ref_col and y0 >= ref_y0:
                    return sorted(cands, key=lambda x: (x["page_0based"], x.get("col", 0), x["y"], x.get("x0", 0.0)))

            if len(line) > MAX_HEADING_LINE_LEN:
                continue

            low = line.lower().strip()
            if any(low.startswith(pref) for pref in BAD_HEADING_PREFIXES):
                continue

            m = SUBHEADING_RE.match(line)
            kind = "sub"
            if not m:
                m = TOPHEADING_RE.match(line)
                kind = "top"
            if not m:
                continue

            num = m.group("num").strip()
            ttl = m.group("title").strip()

            # depth control
            depth = num.count(".")
            if depth > NO_TOC_MAX_DEPTH:
                continue

            # basic title sanity
            if not ttl or len(ttl) < 2:
                continue

            # extra guard: skip reference-like "1. Author ... (2020) ..." lines
            if SKIP_REFERENCE_LIKE_HEADINGS and kind == "top" and _looks_like_reference_entry(ttl):
                continue

            key = f"{p0}|{col}|{round(y0,1)}|{num}|{_norm(ttl)}"
            if key in seen:
                continue
            seen.add(key)

            cands.append({
                "num": num,
                "title": ttl,
                "page_0based": p0,
                "page_1based": p0 + 1,
                "y": float(y0),
                "x0": float(x0),
                "col": int(col),
                "kind": kind,
            })

            if len(cands) >= NO_TOC_MAX_HEADINGS:
                break

    cands.sort(key=lambda x: (x["page_0based"], x.get("col", 0), x["y"], x.get("x0", 0.0)))
    return cands


def _build_entries_from_numbered_headings(cands: List[Dict[str, Any]]) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    entries: List[Dict[str, Any]] = []
    stack_titles: List[str] = []
    stack_ids: List[int] = []

    for toc_id, h in enumerate(cands):
        num = h["num"]
        ttl = h["title"]
        level = num.count(".")  # 0 for "1", 1 for "1.1", ...

        # normalize stack
        stack_titles = stack_titles[:level]
        stack_ids = stack_ids[:level]
        parent_toc_id = stack_ids[-1] if stack_ids else None

        full_title = f"{num} {ttl}".strip()
        stack_titles.append(full_title)
        stack_ids.append(toc_id)

        entries.append({
            "toc_id": toc_id,
            "level": level,
            "title": full_title,
            "titles_path": stack_titles.copy(),
            "toc_path_ids": stack_ids.copy(),
            "parent_toc_id": parent_toc_id,
            "page_0based": h["page_0based"],
            "page_1based": h["page_1based"],
            "y": float(h["y"]),
            "y_snapped": float(h["y"]),
            "col": int(h.get("col", 0)),
            "x0": float(h.get("x0", 0.0)),
            "synthetic": True,
            "synthetic_source": "text_numbered_headings",
            "synthetic_num": num,
        })

    # children map
    by_id = {e["toc_id"]: e for e in entries}
    children_map: Dict[int, List[int]] = {}
    for e in entries:
        pid = e.get("parent_toc_id")
        if pid is not None:
            children_map.setdefault(int(pid), []).append(int(e["toc_id"]))
    for pid, kids in children_map.items():
        kids.sort()
        by_id[pid]["children_toc_ids"] = kids
        by_id[pid]["first_child_toc_id"] = kids[0]

    # ensure entries are ordered in reading order for end-anchor computation
    entries.sort(key=lambda e: (e["page_0based"], int(e.get("col", 0)), float(e.get("y_snapped", e.get("y", 0.0))), float(e.get("x0", 0.0)), e["toc_id"]))

    tree = _build_tree(entries)
    return entries, tree


# ============================================================
# Per-PDF extractor
# ============================================================

def extract_pdf_to_json(source: Dict[str, Any]) -> Dict[str, Any]:
    pdf_path = Path(source["path"])
    source_id = source["source_id"]
    work_id = source["work_id"]

    out_dir = PRE_RAG_DIR / source_id
    out_dir.mkdir(parents=True, exist_ok=True)

    sections_text_dir = out_dir / "sections_text"
    sections_text_dir.mkdir(parents=True, exist_ok=True)

    toc_json_path = out_dir / "toc.json"
    sections_jsonl_path = out_dir / "sections.jsonl"

    info = {
        "unit": UNIT_CODE,
        "work_id": work_id,
        "source_id": source_id,
        "source_file": pdf_path.name,
        "pdf_path": str(pdf_path),
    }

    structured_from_text_headings = False

    with fitz.open(str(pdf_path)) as doc:
        info["page_count"] = doc.page_count

        # 1) Try real PDF ToC/bookmarks
        entries, toc_tree, flip_y = _build_entries_with_paths_and_anchors(doc)

        # 2) If no ToC: optionally structure from numbered headings
        if not entries and STRUCTURE_WHEN_NO_TOC:
            cands = _scan_numbered_headings_whole_doc(doc)
            if len(cands) >= NO_TOC_MIN_HEADINGS:
                entries, toc_tree = _build_entries_from_numbered_headings(cands)
                flip_y = False
                structured_from_text_headings = True

        # ------------------------------------------------------------
        # CASE A: STILL no entries → fallback per-page export
        # ------------------------------------------------------------
        if not entries:
            doc_kind = "publication"
            flip_y_used = bool(flip_y)

            toc_payload = {
                "info": info,
                "doc_kind": doc_kind,
                "toc_entries_count": 0,
                "flip_y_used": flip_y_used,
                "structured_from_text_headings": False,
                "toc_tree": [],
                "flat_entries": [],
                "augmented_entries_count": 0,
                "augmented_tree": [],
                "augmented_flat_entries": [],
            }
            toc_json_path.write_text(json.dumps(toc_payload, ensure_ascii=False, indent=2), encoding="utf-8")

            written = 0
            warnings = 0

            with open(sections_jsonl_path, "w", encoding="utf-8") as f:
                for pno in range(doc.page_count):
                    page = doc.load_page(pno)
                    text = _extract_page_text_column_aware(page).strip()
                    if not text:
                        continue
                    seg = f"page_{pno+1:04d}"
                    leaf_dir = sections_text_dir / seg
                    leaf_dir.mkdir(parents=True, exist_ok=True)
                    txt_path = leaf_dir / "section.txt"
                    txt_path.write_text(text, encoding="utf-8")

                    rec = {
                        **info,
                        "doc_kind": doc_kind,
                        "flip_y_used": flip_y_used,

                        "toc_id": -1,
                        "level": 0,
                        "title": f"Page {pno+1}",
                        "titles_path": ["Publication / Unstructured", f"Page {pno+1}"],
                        "toc_path_ids": [-1],

                        "parent_toc_id": None,
                        "has_children_effective": False,
                        "has_toc_children": False,
                        "auto_children_count": 0,

                        "start": {"page_0based": pno, "page_1based": pno + 1, "y": 0.0},
                        "end_main": {"page_0based": pno, "page_1based": pno + 1, "y": None},
                        "end_full": {"page_0based": pno, "page_1based": pno + 1, "y": None},

                        "synthetic": False,
                        "text_path_main": str(txt_path),
                        "text_path_full_with_children": str(txt_path),

                        "text_chars_main": len(text),
                        "text_words_main": len(text.split()) if text else 0,
                        "text_preview_main": text[:MAX_PREVIEW_CHARS],

                        "next_peer_title": "",
                        "ok_start": True,
                        "ok_end": True,
                    }
                    f.write(json.dumps(rec, ensure_ascii=False) + "\n")
                    written += 1

            return {
                **info,
                "doc_kind": doc_kind,
                "toc_json": str(toc_json_path),
                "sections_jsonl": str(sections_jsonl_path),
                "sections_written": written,
                "warnings": warnings,
                "flip_y_used": flip_y_used,
                "structured_from_text_headings": False,
                "out_dir": str(out_dir),
            }

        # ------------------------------------------------------------
        # CASE B: We have entries (real ToC OR structured from headings)
        # ------------------------------------------------------------
        _compute_end_anchors(entries, doc)
        doc_kind = "book" if (not structured_from_text_headings and len(entries) >= 6) else "publication"
        flip_y_used = bool(flip_y)

        # We'll build augmented entries (original + synthetic auto children)
        augmented_entries: List[Dict[str, Any]] = [dict(e) for e in entries]
        next_toc_id = max(e["toc_id"] for e in entries) + 1

        written = 0
        warnings = 0

        with open(sections_jsonl_path, "w", encoding="utf-8") as f:
            for e in entries:
                toc_id = int(e["toc_id"])
                title = e["title"]
                titles_path = e["titles_path"]
                toc_path_ids = e["toc_path_ids"]
                level = int(e["level"])

                # build directory tree
                cur_dir = sections_text_dir
                for aid, atitle in zip(toc_path_ids, titles_path):
                    cur_dir = cur_dir / _safe_seg(int(aid), atitle)
                cur_dir.mkdir(parents=True, exist_ok=True)

                # anchors
                start_anchor = _anchor_of_entry(e)
                end_full = e.get("end_full", None)

                # children?
                toc_children = e.get("children_toc_ids", [])
                has_toc_children = bool(toc_children)

                # -------------------------
                # AUTO: discover numbered subsections
                # (disable if we already structured whole doc from headings)
                # -------------------------
                discovered: List[Dict[str, Any]] = []
                if (not structured_from_text_headings) and DISCOVER_NUMBERED_SUBSECTIONS and (not DISCOVER_ONLY_IF_NO_TOC_CHILDREN or not has_toc_children):
                    parent_prefix = _extract_number_prefix_from_title(title)
                    discovered = _find_numbered_subheadings_in_range(
                        doc=doc,
                        start_anchor=start_anchor,
                        end_anchor=end_full,
                        parent_prefix=parent_prefix
                    )

                use_auto_children = (len(discovered) > 0) and (not has_toc_children)

                # effective intro end:
                if use_auto_children:
                    first = discovered[0]
                    end_intro = (first["page_0based"], first["y"], int(first.get("col", 0)))
                    has_children_effective = True
                else:
                    end_intro = e.get("end_intro", end_full)
                    has_children_effective = has_toc_children

                # extract main text (intro if parent-effective, else leaf full)
                if has_children_effective:
                    main_text = _extract_between_anchors(doc, start_anchor, end_intro)
                    main_path = cur_dir / "intro.txt"
                else:
                    main_text = _extract_between_anchors(doc, start_anchor, end_full)
                    main_path = cur_dir / "section.txt"

                main_text = (main_text or "").strip()
                if main_text:
                    main_path.write_text(main_text, encoding="utf-8")

                # always write full-with-children for inspection
                full_with_children_txt = _extract_between_anchors(doc, start_anchor, end_full)
                full_with_children_txt = (full_with_children_txt or "").strip()

                full_path = cur_dir / "full_with_children.txt"
                if full_with_children_txt:
                    full_path.write_text(full_with_children_txt, encoding="utf-8")

                # diagnostics
                next_peer_title = (e.get("next_peer_title") or "").strip()
                checks = _boundary_check(main_text, title, next_peer_title if next_peer_title else None)
                if not (checks["ok_start"] and checks["ok_end"]):
                    warnings += 1

                # record parent row
                rec = {
                    **info,
                    "doc_kind": doc_kind,
                    "flip_y_used": flip_y_used,
                    "structured_from_text_headings": structured_from_text_headings,

                    "toc_id": toc_id,
                    "level": level,
                    "title": title,
                    "titles_path": titles_path,
                    "toc_path_ids": toc_path_ids,
                    "parent_toc_id": e.get("parent_toc_id"),

                    "start": {"page_0based": int(start_anchor[0]), "page_1based": int(start_anchor[0]) + 1, "y": float(start_anchor[1])},
                    "end_main": None if end_intro is None else {"page_0based": int(end_intro[0]), "page_1based": int(end_intro[0]) + 1, "y": float(end_intro[1])},
                    "end_full": None if end_full is None else {"page_0based": int(end_full[0]), "page_1based": int(end_full[0]) + 1, "y": float(end_full[1])},

                    "has_children_effective": bool(has_children_effective),
                    "has_toc_children": bool(has_toc_children),
                    "auto_children_count": len(discovered) if use_auto_children else 0,

                    "synthetic": bool(e.get("synthetic", False)),
                    "synthetic_source": e.get("synthetic_source", ""),
                    "synthetic_num": e.get("synthetic_num", ""),

                    "text_path_main": str(main_path) if main_text else "",
                    "text_path_full_with_children": str(full_path) if full_with_children_txt else "",

                    "text_chars_main": len(main_text),
                    "text_words_main": len(main_text.split()) if main_text else 0,
                    "text_preview_main": main_text[:MAX_PREVIEW_CHARS],

                    "next_peer_title": next_peer_title,
                    **checks,
                }
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")
                written += 1

                # Emit synthetic children (only when ToC had none)
                if use_auto_children:
                    for idx, h in enumerate(discovered):
                        child_start = (h["page_0based"], h["y"], int(h.get("col", 0)))
                        if idx < len(discovered) - 1:
                            nxt = discovered[idx + 1]
                            child_end = (nxt["page_0based"], nxt["y"], int(nxt.get("col", 0)))
                        else:
                            child_end = end_full

                        child_title = f"{h['num']} {h['title']}".strip()
                        child_toc_id = next_toc_id
                        next_toc_id += 1

                        child_dir = cur_dir / _safe_seg(child_toc_id, child_title)
                        child_dir.mkdir(parents=True, exist_ok=True)

                        child_text = _extract_between_anchors(doc, child_start, child_end)
                        child_text = (child_text or "").strip()

                        child_path = child_dir / "section.txt"
                        if child_text:
                            child_path.write_text(child_text, encoding="utf-8")

                        c_checks = _boundary_check(child_text, child_title, None)

                        child_entry = {
                            "toc_id": child_toc_id,
                            "level": level + 1,
                            "title": child_title,
                            "titles_path": titles_path + [child_title],
                            "toc_path_ids": toc_path_ids + [child_toc_id],
                            "parent_toc_id": toc_id,
                            "page_0based": int(child_start[0]),
                            "page_1based": int(child_start[0]) + 1,
                            "y": float(child_start[1]),
                            "y_snapped": float(child_start[1]),
                            "col": int(child_start[2]),
                            "end_full": child_end,
                            "end_intro": child_end,
                            "next_peer_title": "",
                            "children_toc_ids": [],
                            "first_child_toc_id": None,
                            "synthetic": True,
                            "synthetic_source": "auto_numbered_subheadings",
                            "synthetic_num": h["num"],
                        }
                        augmented_entries.append(child_entry)

                        child_rec = {
                            **info,
                            "doc_kind": doc_kind,
                            "flip_y_used": flip_y_used,
                            "structured_from_text_headings": structured_from_text_headings,

                            "toc_id": child_toc_id,
                            "level": level + 1,
                            "title": child_title,
                            "titles_path": titles_path + [child_title],
                            "toc_path_ids": toc_path_ids + [child_toc_id],
                            "parent_toc_id": toc_id,

                            "start": {"page_0based": int(child_start[0]), "page_1based": int(child_start[0]) + 1, "y": float(child_start[1])},
                            "end_main": None if child_end is None else {"page_0based": int(child_end[0]), "page_1based": int(child_end[0]) + 1, "y": float(child_end[1])},
                            "end_full": None if child_end is None else {"page_0based": int(child_end[0]), "page_1based": int(child_end[0]) + 1, "y": float(child_end[1])},

                            "has_children_effective": False,
                            "has_toc_children": False,
                            "auto_children_count": 0,

                            "synthetic": True,
                            "synthetic_source": "auto_numbered_subheadings",
                            "synthetic_from_parent_toc_id": toc_id,
                            "synthetic_num": h["num"],

                            "text_path_main": str(child_path) if child_text else "",
                            "text_chars_main": len(child_text),
                            "text_words_main": len(child_text.split()) if child_text else 0,
                            "text_preview_main": child_text[:MAX_PREVIEW_CHARS],

                            **c_checks,
                        }
                        f.write(json.dumps(child_rec, ensure_ascii=False) + "\n")
                        written += 1

        # rebuild augmented tree
        augmented_entries_sorted = sorted(
            augmented_entries,
            key=lambda e: (
                e["page_0based"],
                int(e.get("col", 0)),
                float(e.get("y_snapped", e.get("y", 0.0))),
                float(e.get("x0", 0.0)),
                e["toc_id"],
            )
        )

        by_aug_id = {e["toc_id"]: dict(e) for e in augmented_entries_sorted}
        aug_root: List[Dict[str, Any]] = []
        for e in augmented_entries_sorted:
            node = by_aug_id[e["toc_id"]]
            node.setdefault("children", [])
            pid = e.get("parent_toc_id")
            if pid is None:
                aug_root.append(node)
            else:
                parent = by_aug_id.get(pid)
                if parent is not None:
                    parent.setdefault("children", []).append(node)

        toc_payload = {
            "info": info,
            "doc_kind": doc_kind,
            "toc_entries_count": len(entries),
            "flip_y_used": flip_y_used,
            "structured_from_text_headings": structured_from_text_headings,

            "toc_tree": toc_tree,
            "flat_entries": entries,

            "augmented_entries_count": len(augmented_entries_sorted),
            "augmented_tree": aug_root,
            "augmented_flat_entries": augmented_entries_sorted,

            "auto_subsections_enabled": bool(DISCOVER_NUMBERED_SUBSECTIONS),
            "auto_only_when_no_toc_children": bool(DISCOVER_ONLY_IF_NO_TOC_CHILDREN),
            "structure_when_no_toc": bool(STRUCTURE_WHEN_NO_TOC),

            "column_aware_order": bool(ENABLE_COLUMN_AWARE_ORDER),
            "stop_at_references": bool(STOP_AT_REFERENCES),
        }
        toc_json_path.write_text(json.dumps(toc_payload, ensure_ascii=False, indent=2), encoding="utf-8")

        return {
            **info,
            "doc_kind": doc_kind,
            "toc_json": str(toc_json_path),
            "sections_jsonl": str(sections_jsonl_path),
            "sections_written": written,
            "warnings": warnings,
            "flip_y_used": flip_y_used,
            "structured_from_text_headings": structured_from_text_headings,
            "out_dir": str(out_dir),
        }


# ============================================================
# Run on ALL sources in manifest (PDF only)
# ============================================================

manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
sources = manifest.get("sources", [])
print(f"Loaded {len(sources)} sources from: {MANIFEST_PATH}")

results: List[Dict[str, Any]] = []
skipped = 0
failed = 0

for s in sources:
    ext = (s.get("ext") or Path(s["path"]).suffix.lower())
    if ext != ".pdf":
        skipped += 1
        continue
    try:
        r = extract_pdf_to_json(s)
        results.append(r)
        print(
            f"✅ {Path(s['path']).name} | wrote={r['sections_written']} | warnings={r['warnings']} "
            f"| flip_y={r.get('flip_y_used', False)} | no_toc_structured={r.get('structured_from_text_headings', False)}"
        )
    except Exception as e:
        failed += 1
        print(f"❌ FAILED {s.get('path')}: {e}")

summary_path = PRE_RAG_DIR / "pdf_extraction_summary.json"
summary_path.write_text(
    json.dumps({"unit": UNIT_CODE, "results": results, "skipped_non_pdf": skipped, "failed": failed}, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("-" * 70)
print(f"Done. Extracted PDFs: {len(results)} | Failed: {failed} | Skipped non-PDF: {skipped}")
print(f"Summary: {summary_path}")
print(f"Per-source outputs: {PRE_RAG_DIR}")


Loaded 4 sources from: data_output/MBAI5004/sources_manifest.json
✅ Ethical AI in Information Technology Navigating Bias, Privacy, Transparency,.pdf | wrote=65 | warnings=2 | flip_y=False | no_toc_structured=True
✅ Ethics Of Artificial Intelligence -- S_ Matthew Liao.pdf | wrote=225 | warnings=17 | flip_y=False | no_toc_structured=False
✅ Ethics of Artificial Intelligence Case Studies.pdf | wrote=90 | warnings=8 | flip_y=True | no_toc_structured=False
✅ Ethics, Governance, and Policies in Artificial Intelligence -- Luciano Floridi.pdf | wrote=260 | warnings=34 | flip_y=True | no_toc_structured=False
----------------------------------------------------------------------
Done. Extracted PDFs: 4 | Failed: 0 | Skipped non-PDF: 0
Summary: data_output/MBAI5004/pre_rag_pdf_sections/pdf_extraction_summary.json
Per-source outputs: data_output/MBAI5004/pre_rag_pdf_sections


In [4]:
# ===========================⭐⭐⭐=================================
# CELL X — MULTI-SOURCE PDF PRE-EXTRACTION
#   (Anchors + Parent Intro + AUTO Subsections + NO-TOC STRUCTURING)
#
# Writes per-source folder (by source_id):
#   - toc.json  (includes augmented_flat_entries + augmented_tree)
#   - sections.jsonl
#   - sections_text/<tree...>/(intro.txt | section.txt | full_with_children.txt)
#
# Key fixes:
#   ✅ Always returns "flip_y_used" in ALL branches (no KeyError in print)
#   ✅ Optional: if PDF has NO ToC, build synthetic section tree from numbered headings
# ============================================================

from __future__ import annotations
import json
import re
import hashlib
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
from difflib import SequenceMatcher

import fitz  # PyMuPDF


# -----------------------------
# Uses your globals from Cell 1
# -----------------------------
MANIFEST_PATH = Path(MANIFEST_PATH)
OUTPUT_ROOT = Path(OUTPUT_ROOT)

PRE_RAG_DIR = OUTPUT_ROOT / "pre_rag_pdf_sections"
PRE_RAG_DIR.mkdir(parents=True, exist_ok=True)


# -----------------------------
# Extraction options
# -----------------------------
SORT_BLOCKS = True
DROP_HEADERS_FOOTERS = True
HEADER_PX = 50.0
FOOTER_PX = 55.0

TITLE_MATCH_RATIO = 0.74
TITLE_SNAP_Y_WINDOW_UP = 120.0
TITLE_SNAP_Y_WINDOW_DOWN = 380.0

ANCHOR_Y_PROX = 90.0
SCORE_SAMPLE_N = 30

MAX_PREVIEW_CHARS = 2500

# -----------------------------
# Auto subsection discovery (within a ToC parent range)
# -----------------------------
DISCOVER_NUMBERED_SUBSECTIONS = True
DISCOVER_ONLY_IF_NO_TOC_CHILDREN = True   # conservative default (recommended)
MAX_HEADING_LINE_LEN = 140

# 1.1. Introduction   OR   1.1 Introduction   OR   1.2.3 Some title
SUBHEADING_RE = re.compile(r"^\s*(?P<num>\d+(?:\.\d+)+)\.?\s+(?P<title>.+?)\s*$")

# top-level: 1 Introduction OR 2. Methods
TOPHEADING_RE = re.compile(r"^\s*(?P<num>\d+)\.?\s+(?P<title>.+?)\s*$")

# avoid capturing Fig. 2.1, Table 3.2, etc.
BAD_HEADING_PREFIXES = ("fig", "figure", "table", "eq", "equation")

# -----------------------------
# NEW: No-ToC structuring from numbered headings
# -----------------------------
STRUCTURE_WHEN_NO_TOC = True
NO_TOC_MIN_HEADINGS = 5          # if fewer than this, fallback to per-page export
NO_TOC_MAX_HEADINGS = 600        # safety cap
NO_TOC_MAX_DEPTH = 5             # max dots depth (e.g., 1.2.3.4.5)


# ============================================================
# Helpers
# ============================================================

def _sha1(s: str) -> str:
    return hashlib.sha1(s.encode("utf-8", errors="ignore")).hexdigest()

def _norm(s: str) -> str:
    s = (s or "").lower()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"[^\w\s]", "", s)
    return s.strip()

def _slug(s: str, max_len: int = 42) -> str:
    s = (s or "")
    s = s.replace("’", "'")
    s = re.sub(r"\s+", "_", s.strip())
    s = re.sub(r"[^A-Za-z0-9_\-]+", "", s)
    return (s or "untitled")[:max_len]

def _safe_seg(toc_id: int, title: str) -> str:
    h = _sha1(title)[:6]
    return f"toc{toc_id:04d}__{_slug(title)}__{h}"

def _similar(a: str, b: str) -> float:
    return SequenceMatcher(None, _norm(a), _norm(b)).ratio()

def _block_list(page: fitz.Page) -> List[Tuple]:
    blocks = page.get_text("blocks", sort=SORT_BLOCKS)
    blocks = [b for b in blocks if b and len(b) >= 5]
    if DROP_HEADERS_FOOTERS:
        h = float(page.rect.height)
        out = []
        for b in blocks:
            y0, y1 = float(b[1]), float(b[3])
            if y1 < HEADER_PX:
                continue
            if y0 > (h - FOOTER_PX):
                continue
            out.append(b)
        blocks = out
    return blocks

def _find_heading_block_y(page: fitz.Page, y_hint: float, title: str) -> Optional[float]:
    blocks = _block_list(page)
    y0_min = max(0.0, y_hint - TITLE_SNAP_Y_WINDOW_UP)
    y0_max = min(float(page.rect.height), y_hint + TITLE_SNAP_Y_WINDOW_DOWN)

    best = None
    best_score = 0.0
    for (x0, y0, x1, y1, text, *rest) in blocks:
        y0f = float(y0)
        if y0f < y0_min or y0f > y0_max:
            continue
        t = (text or "").strip()
        if not t:
            continue
        score = max(_similar(t, title), 1.0 if _norm(title) in _norm(t) else 0.0)
        if score > best_score:
            best_score = score
            best = y0f

    if best is not None and best_score >= TITLE_MATCH_RATIO:
        return best
    return None

def _extract_blocks_between(page: fitz.Page, y_start: float, y_end: Optional[float]) -> str:
    h = float(page.rect.height)
    y_start = max(0.0, min(y_start, h))
    if y_end is not None:
        y_end = max(0.0, min(y_end, h))
        if y_end <= y_start:
            return ""

    blocks = _block_list(page)
    out: List[str] = []
    for (x0, y0, x1, y1, text, *rest) in blocks:
        y0f, y1f = float(y0), float(y1)
        if y1f <= y_start:
            continue
        if y_end is not None and y0f >= y_end:
            continue
        t = (text or "").strip()
        if t:
            out.append(t)

    joined = "\n".join(out)
    joined = re.sub(r"\n{3,}", "\n\n", joined).strip()
    return joined

def _extract_range(
    doc: fitz.Document,
    start_page0: int,
    start_y: float,
    end_page0: Optional[int],
    end_y: Optional[float],
) -> str:
    n_pages = doc.page_count
    start_page0 = max(0, min(start_page0, n_pages - 1))

    if end_page0 is None:
        end_page0 = n_pages - 1
        end_y = None
    else:
        end_page0 = max(0, min(end_page0, n_pages - 1))

    if end_page0 < start_page0:
        end_page0 = start_page0
        end_y = None

    parts: List[str] = []
    if start_page0 == end_page0:
        parts.append(_extract_blocks_between(doc.load_page(start_page0), start_y, end_y))
    else:
        parts.append(_extract_blocks_between(doc.load_page(start_page0), start_y, None))
        for p in range(start_page0 + 1, end_page0):
            parts.append(_extract_blocks_between(doc.load_page(p), 0.0, None))
        parts.append(_extract_blocks_between(doc.load_page(end_page0), 0.0, end_y))

    text = "\n\n".join([p for p in parts if p.strip()]).strip()
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    return text

def _boundary_check(section_text: str, this_title: str, next_title: Optional[str]) -> Dict[str, Any]:
    head = (section_text or "")[:1800]
    ok_start = (_norm(this_title) in _norm(head)) or (_similar(head, this_title) >= 0.35)
    ok_end = True
    if next_title:
        ok_end = (_norm(next_title) not in _norm(head))
    return {"ok_start": bool(ok_start), "ok_end": bool(ok_end)}


# ============================================================
# ToC extraction + coordinate orientation selection
# ============================================================

def _get_detailed_toc(doc: fitz.Document) -> List[List[Any]]:
    return doc.get_toc(simple=False)

def _to_y_float(to: Any) -> float:
    # dest["to"] can be fitz.Point OR tuple/list OR something else
    if to is None:
        return 0.0
    if hasattr(to, "y"):
        return float(getattr(to, "y", 0.0))
    if isinstance(to, (list, tuple)) and len(to) >= 2:
        try:
            return float(to[1])
        except Exception:
            return 0.0
    return 0.0

def _score_orientation(doc: fitz.Document, toc_items: List[List[Any]], flip_y: bool) -> int:
    score = 0
    cache_blocks: Dict[int, List[Tuple]] = {}

    for row in toc_items[:SCORE_SAMPLE_N]:
        if len(row) < 4:
            continue
        title, page1, dest = row[1], row[2], row[3]
        if not isinstance(dest, dict):
            continue
        p0 = int(dest.get("page", (page1 - 1 if isinstance(page1, int) else 0)))
        if p0 < 0 or p0 >= doc.page_count:
            continue

        to = dest.get("to", None)
        if to is None:
            continue

        page = doc.load_page(p0)
        h = float(page.rect.height)
        y = _to_y_float(to)
        y = (h - y) if flip_y else y

        if p0 not in cache_blocks:
            cache_blocks[p0] = _block_list(page)

        title_s = (title or "").strip()
        if not title_s:
            continue

        found = False
        for (x0, y0, x1, y1, text, *rest) in cache_blocks[p0]:
            if abs(float(y0) - y) > ANCHOR_Y_PROX:
                continue
            t = (text or "").strip()
            if not t:
                continue
            if _norm(title_s) in _norm(t) or _similar(t, title_s) >= TITLE_MATCH_RATIO:
                found = True
                break

        if found:
            score += 1

    return score

def _build_tree(entries: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    by_id = {e["toc_id"]: {k: e[k] for k in e.keys() if k not in ("parent_toc_id",)} for e in entries}
    root: List[Dict[str, Any]] = []
    for e in entries:
        node = by_id[e["toc_id"]]
        node["children"] = []
        pid = e.get("parent_toc_id")
        if pid is None:
            root.append(node)
        else:
            parent = by_id.get(pid)
            if parent is not None:
                parent["children"].append(node)
    return root

def _anchor_of_entry(e: Dict[str, Any]) -> Tuple[int, float]:
    return int(e["page_0based"]), float(e.get("y_snapped", e.get("y", 0.0)))

def _compute_end_anchors(entries: List[Dict[str, Any]], doc: fitz.Document) -> None:
    n = len(entries)
    for i in range(n):
        cur = entries[i]
        cur_lvl = int(cur["level"])

        end_full: Optional[Tuple[int, float]] = None
        next_peer_title: Optional[str] = None
        for j in range(i + 1, n):
            nxt = entries[j]
            if int(nxt["level"]) <= cur_lvl:
                end_full = _anchor_of_entry(nxt)
                next_peer_title = nxt["title"]
                break

        cur["end_full"] = end_full
        cur["next_peer_title"] = next_peer_title or ""

        kids = cur.get("children_toc_ids", [])
        if kids:
            first_child = next((x for x in entries if x["toc_id"] == kids[0]), None)
            cur["end_intro"] = _anchor_of_entry(first_child) if first_child else end_full
        else:
            cur["end_intro"] = end_full

def _build_entries_with_paths_and_anchors(doc: fitz.Document) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]], bool]:
    toc_items = _get_detailed_toc(doc)
    if not toc_items:
        return [], [], False

    raw_score = _score_orientation(doc, toc_items, flip_y=False)
    flip_score = _score_orientation(doc, toc_items, flip_y=True)
    flip_y = (flip_score > raw_score)

    entries: List[Dict[str, Any]] = []
    stack_titles: List[str] = []
    stack_ids: List[int] = []

    toc_id = 0
    for row in toc_items:
        if len(row) < 3:
            continue
        lvl, title, page1 = row[0], row[1], row[2]
        dest = row[3] if len(row) >= 4 else None

        if not isinstance(lvl, int) or lvl < 1:
            continue
        title = (title or "").strip()
        if not title:
            continue
        if not isinstance(page1, int) or page1 < 1 or page1 > doc.page_count:
            continue

        if isinstance(dest, dict) and "page" in dest:
            page0 = int(dest["page"])
        else:
            page0 = page1 - 1
        page0 = max(0, min(page0, doc.page_count - 1))

        page = doc.load_page(page0)
        h = float(page.rect.height)

        y = 0.0
        if isinstance(dest, dict) and dest.get("to", None) is not None:
            y = _to_y_float(dest["to"])
            if flip_y:
                y = (h - y)

        level0 = lvl - 1

        stack_titles = stack_titles[:level0]
        stack_ids = stack_ids[:level0]
        parent_toc_id = stack_ids[-1] if stack_ids else None

        stack_titles.append(title)
        stack_ids.append(toc_id)

        e = {
            "toc_id": toc_id,
            "level": level0,
            "title": title,
            "titles_path": stack_titles.copy(),
            "toc_path_ids": stack_ids.copy(),
            "parent_toc_id": parent_toc_id,
            "page_0based": page0,
            "page_1based": page0 + 1,
            "y": float(y),
            "synthetic": False,
        }
        entries.append(e)
        toc_id += 1

    entries.sort(key=lambda e: (e["page_0based"], e["y"], e["toc_id"]))

    by_id = {e["toc_id"]: e for e in entries}
    children_map: Dict[int, List[int]] = {}
    for e in entries:
        pid = e.get("parent_toc_id")
        if pid is not None:
            children_map.setdefault(int(pid), []).append(int(e["toc_id"]))
    for pid, kids in children_map.items():
        kids.sort()
        by_id[pid]["children_toc_ids"] = kids
        by_id[pid]["first_child_toc_id"] = kids[0]

    # snap y to nearest real heading block y
    for e in entries:
        p0 = e["page_0based"]
        page = doc.load_page(p0)
        snapped = _find_heading_block_y(page, e["y"], e["title"])
        e["y_snapped"] = float(snapped) if snapped is not None else float(e["y"])

    return entries, _build_tree(entries), flip_y


# ============================================================
# Numbered subheadings discovery inside a range (dict lines+bbox)
# ============================================================

def _iter_lines_with_bbox(page: fitz.Page) -> List[Tuple[float, float, float, float, str]]:
    d = page.get_text("dict")
    lines_out: List[Tuple[float, float, float, float, str]] = []
    h = float(page.rect.height)

    for b in d.get("blocks", []):
        if b.get("type", 0) != 0:
            continue
        for ln in b.get("lines", []):
            bbox = ln.get("bbox", None)
            if not bbox or len(bbox) != 4:
                continue
            x0, y0, x1, y1 = map(float, bbox)

            if DROP_HEADERS_FOOTERS:
                if y1 < HEADER_PX:
                    continue
                if y0 > (h - FOOTER_PX):
                    continue

            spans = ln.get("spans", [])
            txt = "".join([(sp.get("text") or "") for sp in spans]).strip()
            txt = re.sub(r"\s+", " ", txt).strip()
            if txt:
                lines_out.append((x0, y0, x1, y1, txt))
    return lines_out

def _extract_number_prefix_from_title(title: str) -> Optional[str]:
    """
    If title starts with a number prefix like:
      "1 Ethical Learning" -> "1"
      "1.2 A Social Perspective" -> "1.2"
      "2.3.1 Something" -> "2.3.1"
    """
    m = re.match(r"^\s*(\d+(?:\.\d+)*)\b", title or "")
    return m.group(1) if m else None

def _find_numbered_subheadings_in_range(
    doc: fitz.Document,
    start_anchor: Tuple[int, float],
    end_anchor: Optional[Tuple[int, float]],
    parent_prefix: Optional[str],
) -> List[Dict[str, Any]]:
    s_p0, s_y = start_anchor
    if end_anchor is None:
        e_p0, e_y = (doc.page_count - 1, None)
    else:
        e_p0, e_y = (end_anchor[0], end_anchor[1])

    cands: List[Dict[str, Any]] = []
    seen = set()

    for p0 in range(s_p0, e_p0 + 1):
        page = doc.load_page(p0)
        for (x0, y0, x1, y1, line) in _iter_lines_with_bbox(page):
            if p0 == s_p0 and y1 <= s_y:
                continue
            if end_anchor is not None and p0 == e_p0 and e_y is not None and y0 >= e_y:
                continue

            if len(line) > MAX_HEADING_LINE_LEN:
                continue

            low = line.lower().strip()
            if any(low.startswith(pref) for pref in BAD_HEADING_PREFIXES):
                continue

            m = SUBHEADING_RE.match(line)
            if not m:
                continue

            num = m.group("num").strip()
            ttl = m.group("title").strip()

            # prefix filter:
            #   parent "1"   -> accept "1.x"
            #   parent "1.1" -> accept "1.1.x"
            if parent_prefix:
                must = parent_prefix + "."
                if not num.startswith(must):
                    continue

            key = f"{p0}|{round(y0,1)}|{num}|{_norm(ttl)}"
            if key in seen:
                continue
            seen.add(key)

            cands.append({
                "num": num,
                "title": ttl,
                "page_0based": p0,
                "page_1based": p0 + 1,
                "y": float(y0),
            })

    cands.sort(key=lambda x: (x["page_0based"], x["y"]))
    return cands


# ============================================================
# NEW: Build synthetic entries from numbered headings when no ToC
# ============================================================

def _scan_numbered_headings_whole_doc(doc: fitz.Document) -> List[Dict[str, Any]]:
    cands: List[Dict[str, Any]] = []
    seen = set()

    for p0 in range(doc.page_count):
        page = doc.load_page(p0)
        for (x0, y0, x1, y1, line) in _iter_lines_with_bbox(page):
            if len(line) > MAX_HEADING_LINE_LEN:
                continue

            low = line.lower().strip()
            if any(low.startswith(pref) for pref in BAD_HEADING_PREFIXES):
                continue

            m = SUBHEADING_RE.match(line)
            kind = "sub"
            if not m:
                m = TOPHEADING_RE.match(line)
                kind = "top"
            if not m:
                continue

            num = m.group("num").strip()
            ttl = m.group("title").strip()

            # depth control
            depth = num.count(".")
            if depth > NO_TOC_MAX_DEPTH:
                continue

            # basic title sanity
            if not ttl or len(ttl) < 2:
                continue

            key = f"{p0}|{round(y0,1)}|{num}|{_norm(ttl)}"
            if key in seen:
                continue
            seen.add(key)

            cands.append({
                "num": num,
                "title": ttl,
                "page_0based": p0,
                "page_1based": p0 + 1,
                "y": float(y0),
                "kind": kind,
            })

            if len(cands) >= NO_TOC_MAX_HEADINGS:
                break

    cands.sort(key=lambda x: (x["page_0based"], x["y"]))
    return cands

def _build_entries_from_numbered_headings(cands: List[Dict[str, Any]]) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    entries: List[Dict[str, Any]] = []
    stack_titles: List[str] = []
    stack_ids: List[int] = []

    for toc_id, h in enumerate(cands):
        num = h["num"]
        ttl = h["title"]
        level = num.count(".")  # 0 for "1", 1 for "1.1", ...

        # normalize stack
        stack_titles = stack_titles[:level]
        stack_ids = stack_ids[:level]
        parent_toc_id = stack_ids[-1] if stack_ids else None

        full_title = f"{num} {ttl}".strip()
        stack_titles.append(full_title)
        stack_ids.append(toc_id)

        entries.append({
            "toc_id": toc_id,
            "level": level,
            "title": full_title,
            "titles_path": stack_titles.copy(),
            "toc_path_ids": stack_ids.copy(),
            "parent_toc_id": parent_toc_id,
            "page_0based": h["page_0based"],
            "page_1based": h["page_1based"],
            "y": float(h["y"]),
            "y_snapped": float(h["y"]),
            "synthetic": True,
            "synthetic_source": "text_numbered_headings",
            "synthetic_num": num,
        })

    # children map
    by_id = {e["toc_id"]: e for e in entries}
    children_map: Dict[int, List[int]] = {}
    for e in entries:
        pid = e.get("parent_toc_id")
        if pid is not None:
            children_map.setdefault(int(pid), []).append(int(e["toc_id"]))
    for pid, kids in children_map.items():
        kids.sort()
        by_id[pid]["children_toc_ids"] = kids
        by_id[pid]["first_child_toc_id"] = kids[0]

    tree = _build_tree(entries)
    return entries, tree


# ============================================================
# Per-PDF extractor
# ============================================================

def extract_pdf_to_json(source: Dict[str, Any]) -> Dict[str, Any]:
    pdf_path = Path(source["path"])
    source_id = source["source_id"]
    work_id = source["work_id"]

    out_dir = PRE_RAG_DIR / source_id
    out_dir.mkdir(parents=True, exist_ok=True)

    sections_text_dir = out_dir / "sections_text"
    sections_text_dir.mkdir(parents=True, exist_ok=True)

    toc_json_path = out_dir / "toc.json"
    sections_jsonl_path = out_dir / "sections.jsonl"

    info = {
        "unit": UNIT_CODE,
        "work_id": work_id,
        "source_id": source_id,
        "source_file": pdf_path.name,
        "pdf_path": str(pdf_path),
    }

    structured_from_text_headings = False

    with fitz.open(str(pdf_path)) as doc:
        info["page_count"] = doc.page_count

        # 1) Try real PDF ToC/bookmarks
        entries, toc_tree, flip_y = _build_entries_with_paths_and_anchors(doc)

        # 2) If no ToC: optionally structure from numbered headings
        if not entries and STRUCTURE_WHEN_NO_TOC:
            cands = _scan_numbered_headings_whole_doc(doc)
            if len(cands) >= NO_TOC_MIN_HEADINGS:
                entries, toc_tree = _build_entries_from_numbered_headings(cands)
                flip_y = False
                structured_from_text_headings = True

        # ------------------------------------------------------------
        # CASE A: STILL no entries → fallback per-page export
        # ------------------------------------------------------------
        if not entries:
            doc_kind = "publication"
            flip_y_used = bool(flip_y)

            toc_payload = {
                "info": info,
                "doc_kind": doc_kind,
                "toc_entries_count": 0,
                "flip_y_used": flip_y_used,
                "structured_from_text_headings": False,
                "toc_tree": [],
                "flat_entries": [],
                "augmented_entries_count": 0,
                "augmented_tree": [],
                "augmented_flat_entries": [],
            }
            toc_json_path.write_text(json.dumps(toc_payload, ensure_ascii=False, indent=2), encoding="utf-8")

            written = 0
            warnings = 0

            with open(sections_jsonl_path, "w", encoding="utf-8") as f:
                for pno in range(doc.page_count):
                    text = doc.load_page(pno).get_text("text", sort=True).strip()
                    if not text:
                        continue
                    seg = f"page_{pno+1:04d}"
                    leaf_dir = sections_text_dir / seg
                    leaf_dir.mkdir(parents=True, exist_ok=True)
                    txt_path = leaf_dir / "section.txt"
                    txt_path.write_text(text, encoding="utf-8")

                    rec = {
                        **info,
                        "doc_kind": doc_kind,
                        "flip_y_used": flip_y_used,

                        "toc_id": -1,
                        "level": 0,
                        "title": f"Page {pno+1}",
                        "titles_path": ["Publication / Unstructured", f"Page {pno+1}"],
                        "toc_path_ids": [-1],

                        "parent_toc_id": None,
                        "has_children_effective": False,
                        "has_toc_children": False,
                        "auto_children_count": 0,

                        "start": {"page_0based": pno, "page_1based": pno + 1, "y": 0.0},
                        "end_main": {"page_0based": pno, "page_1based": pno + 1, "y": None},
                        "end_full": {"page_0based": pno, "page_1based": pno + 1, "y": None},

                        "synthetic": False,
                        "text_path_main": str(txt_path),
                        "text_path_full_with_children": str(txt_path),

                        "text_chars_main": len(text),
                        "text_words_main": len(text.split()) if text else 0,
                        "text_preview_main": text[:MAX_PREVIEW_CHARS],

                        "next_peer_title": "",
                        "ok_start": True,
                        "ok_end": True,
                    }
                    f.write(json.dumps(rec, ensure_ascii=False) + "\n")
                    written += 1

            return {
                **info,
                "doc_kind": doc_kind,
                "toc_json": str(toc_json_path),
                "sections_jsonl": str(sections_jsonl_path),
                "sections_written": written,
                "warnings": warnings,
                "flip_y_used": flip_y_used,
                "structured_from_text_headings": False,
                "out_dir": str(out_dir),
            }

        # ------------------------------------------------------------
        # CASE B: We have entries (real ToC OR structured from headings)
        # ------------------------------------------------------------
        _compute_end_anchors(entries, doc)
        doc_kind = "book" if (not structured_from_text_headings and len(entries) >= 6) else "publication"
        flip_y_used = bool(flip_y)

        # We'll build augmented entries (original + synthetic auto children)
        augmented_entries: List[Dict[str, Any]] = [dict(e) for e in entries]
        next_toc_id = max(e["toc_id"] for e in entries) + 1

        written = 0
        warnings = 0

        with open(sections_jsonl_path, "w", encoding="utf-8") as f:
            for e in entries:
                toc_id = int(e["toc_id"])
                title = e["title"]
                titles_path = e["titles_path"]
                toc_path_ids = e["toc_path_ids"]
                level = int(e["level"])

                # build directory tree
                cur_dir = sections_text_dir
                for aid, atitle in zip(toc_path_ids, titles_path):
                    cur_dir = cur_dir / _safe_seg(int(aid), atitle)
                cur_dir.mkdir(parents=True, exist_ok=True)

                # anchors
                start_anchor = _anchor_of_entry(e)
                end_full = e.get("end_full", None)

                # children?
                toc_children = e.get("children_toc_ids", [])
                has_toc_children = bool(toc_children)

                # -------------------------
                # AUTO: discover numbered subsections
                # (disable if we already structured whole doc from headings)
                # -------------------------
                discovered: List[Dict[str, Any]] = []
                if (not structured_from_text_headings) and DISCOVER_NUMBERED_SUBSECTIONS and (not DISCOVER_ONLY_IF_NO_TOC_CHILDREN or not has_toc_children):
                    parent_prefix = _extract_number_prefix_from_title(title)
                    discovered = _find_numbered_subheadings_in_range(
                        doc=doc,
                        start_anchor=start_anchor,
                        end_anchor=end_full,
                        parent_prefix=parent_prefix
                    )

                use_auto_children = (len(discovered) > 0) and (not has_toc_children)

                # effective intro end:
                if use_auto_children:
                    first = discovered[0]
                    end_intro = (first["page_0based"], first["y"])
                    has_children_effective = True
                else:
                    end_intro = e.get("end_intro", end_full)
                    has_children_effective = has_toc_children

                # extract main text (intro if parent-effective, else leaf full)
                if has_children_effective:
                    if end_intro is None:
                        main_text = _extract_range(doc, start_anchor[0], start_anchor[1], None, None)
                    else:
                        main_text = _extract_range(doc, start_anchor[0], start_anchor[1], end_intro[0], end_intro[1])
                    main_path = cur_dir / "intro.txt"
                else:
                    if end_full is None:
                        main_text = _extract_range(doc, start_anchor[0], start_anchor[1], None, None)
                    else:
                        main_text = _extract_range(doc, start_anchor[0], start_anchor[1], end_full[0], end_full[1])
                    main_path = cur_dir / "section.txt"

                main_text = (main_text or "").strip()
                if main_text:
                    main_path.write_text(main_text, encoding="utf-8")

                # always write full-with-children for inspection
                if end_full is None:
                    full_with_children_txt = _extract_range(doc, start_anchor[0], start_anchor[1], None, None)
                else:
                    full_with_children_txt = _extract_range(doc, start_anchor[0], start_anchor[1], end_full[0], end_full[1])
                full_with_children_txt = (full_with_children_txt or "").strip()

                full_path = cur_dir / "full_with_children.txt"
                if full_with_children_txt:
                    full_path.write_text(full_with_children_txt, encoding="utf-8")

                # diagnostics
                next_peer_title = (e.get("next_peer_title") or "").strip()
                checks = _boundary_check(main_text, title, next_peer_title if next_peer_title else None)
                if not (checks["ok_start"] and checks["ok_end"]):
                    warnings += 1

                # record parent row
                rec = {
                    **info,
                    "doc_kind": doc_kind,
                    "flip_y_used": flip_y_used,
                    "structured_from_text_headings": structured_from_text_headings,

                    "toc_id": toc_id,
                    "level": level,
                    "title": title,
                    "titles_path": titles_path,
                    "toc_path_ids": toc_path_ids,
                    "parent_toc_id": e.get("parent_toc_id"),

                    "start": {"page_0based": start_anchor[0], "page_1based": start_anchor[0] + 1, "y": start_anchor[1]},
                    "end_main": None if end_intro is None else {"page_0based": end_intro[0], "page_1based": end_intro[0] + 1, "y": end_intro[1]},
                    "end_full": None if end_full is None else {"page_0based": end_full[0], "page_1based": end_full[0] + 1, "y": end_full[1]},

                    "has_children_effective": bool(has_children_effective),
                    "has_toc_children": bool(has_toc_children),
                    "auto_children_count": len(discovered) if use_auto_children else 0,

                    "synthetic": bool(e.get("synthetic", False)),
                    "synthetic_source": e.get("synthetic_source", ""),
                    "synthetic_num": e.get("synthetic_num", ""),

                    "text_path_main": str(main_path) if main_text else "",
                    "text_path_full_with_children": str(full_path) if full_with_children_txt else "",

                    "text_chars_main": len(main_text),
                    "text_words_main": len(main_text.split()) if main_text else 0,
                    "text_preview_main": main_text[:MAX_PREVIEW_CHARS],

                    "next_peer_title": next_peer_title,
                    **checks,
                }
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")
                written += 1

                # Emit synthetic children (only when ToC had none)
                if use_auto_children:
                    for idx, h in enumerate(discovered):
                        child_start = (h["page_0based"], h["y"])
                        if idx < len(discovered) - 1:
                            nxt = discovered[idx + 1]
                            child_end = (nxt["page_0based"], nxt["y"])
                        else:
                            child_end = end_full

                        child_title = f"{h['num']} {h['title']}".strip()
                        child_toc_id = next_toc_id
                        next_toc_id += 1

                        child_dir = cur_dir / _safe_seg(child_toc_id, child_title)
                        child_dir.mkdir(parents=True, exist_ok=True)

                        if child_end is None:
                            child_text = _extract_range(doc, child_start[0], child_start[1], None, None)
                        else:
                            child_text = _extract_range(doc, child_start[0], child_start[1], child_end[0], child_end[1])
                        child_text = (child_text or "").strip()

                        child_path = child_dir / "section.txt"
                        if child_text:
                            child_path.write_text(child_text, encoding="utf-8")

                        c_checks = _boundary_check(child_text, child_title, None)

                        child_entry = {
                            "toc_id": child_toc_id,
                            "level": level + 1,
                            "title": child_title,
                            "titles_path": titles_path + [child_title],
                            "toc_path_ids": toc_path_ids + [child_toc_id],
                            "parent_toc_id": toc_id,
                            "page_0based": child_start[0],
                            "page_1based": child_start[0] + 1,
                            "y": child_start[1],
                            "y_snapped": child_start[1],
                            "end_full": child_end,
                            "end_intro": child_end,
                            "next_peer_title": "",
                            "children_toc_ids": [],
                            "first_child_toc_id": None,
                            "synthetic": True,
                            "synthetic_source": "auto_numbered_subheadings",
                            "synthetic_num": h["num"],
                        }
                        augmented_entries.append(child_entry)

                        child_rec = {
                            **info,
                            "doc_kind": doc_kind,
                            "flip_y_used": flip_y_used,
                            "structured_from_text_headings": structured_from_text_headings,

                            "toc_id": child_toc_id,
                            "level": level + 1,
                            "title": child_title,
                            "titles_path": titles_path + [child_title],
                            "toc_path_ids": toc_path_ids + [child_toc_id],
                            "parent_toc_id": toc_id,

                            "start": {"page_0based": child_start[0], "page_1based": child_start[0] + 1, "y": child_start[1]},
                            "end_main": None if child_end is None else {"page_0based": child_end[0], "page_1based": child_end[0] + 1, "y": child_end[1]},
                            "end_full": None if child_end is None else {"page_0based": child_end[0], "page_1based": child_end[0] + 1, "y": child_end[1]},

                            "has_children_effective": False,
                            "has_toc_children": False,
                            "auto_children_count": 0,

                            "synthetic": True,
                            "synthetic_source": "auto_numbered_subheadings",
                            "synthetic_from_parent_toc_id": toc_id,
                            "synthetic_num": h["num"],

                            "text_path_main": str(child_path) if child_text else "",
                            "text_chars_main": len(child_text),
                            "text_words_main": len(child_text.split()) if child_text else 0,
                            "text_preview_main": child_text[:MAX_PREVIEW_CHARS],

                            **c_checks,
                        }
                        f.write(json.dumps(child_rec, ensure_ascii=False) + "\n")
                        written += 1

        # rebuild augmented tree
        augmented_entries_sorted = sorted(
            augmented_entries,
            key=lambda e: (e["page_0based"], float(e.get("y_snapped", e.get("y", 0.0))), e["toc_id"])
        )

        by_aug_id = {e["toc_id"]: dict(e) for e in augmented_entries_sorted}
        aug_root: List[Dict[str, Any]] = []
        for e in augmented_entries_sorted:
            node = by_aug_id[e["toc_id"]]
            node.setdefault("children", [])
            pid = e.get("parent_toc_id")
            if pid is None:
                aug_root.append(node)
            else:
                parent = by_aug_id.get(pid)
                if parent is not None:
                    parent.setdefault("children", []).append(node)

        toc_payload = {
            "info": info,
            "doc_kind": doc_kind,
            "toc_entries_count": len(entries),
            "flip_y_used": flip_y_used,
            "structured_from_text_headings": structured_from_text_headings,

            "toc_tree": toc_tree,
            "flat_entries": entries,

            "augmented_entries_count": len(augmented_entries_sorted),
            "augmented_tree": aug_root,
            "augmented_flat_entries": augmented_entries_sorted,

            "auto_subsections_enabled": bool(DISCOVER_NUMBERED_SUBSECTIONS),
            "auto_only_when_no_toc_children": bool(DISCOVER_ONLY_IF_NO_TOC_CHILDREN),
            "structure_when_no_toc": bool(STRUCTURE_WHEN_NO_TOC),
        }
        toc_json_path.write_text(json.dumps(toc_payload, ensure_ascii=False, indent=2), encoding="utf-8")

        return {
            **info,
            "doc_kind": doc_kind,
            "toc_json": str(toc_json_path),
            "sections_jsonl": str(sections_jsonl_path),
            "sections_written": written,
            "warnings": warnings,
            "flip_y_used": flip_y_used,
            "structured_from_text_headings": structured_from_text_headings,
            "out_dir": str(out_dir),
        }


# ============================================================
# Run on ALL sources in manifest (PDF only)
# ============================================================

manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
sources = manifest.get("sources", [])
print(f"Loaded {len(sources)} sources from: {MANIFEST_PATH}")

results: List[Dict[str, Any]] = []
skipped = 0
failed = 0

for s in sources:
    ext = (s.get("ext") or Path(s["path"]).suffix.lower())
    if ext != ".pdf":
        skipped += 1
        continue
    try:
        r = extract_pdf_to_json(s)
        results.append(r)
        print(
            f"✅ {Path(s['path']).name} | wrote={r['sections_written']} | warnings={r['warnings']} "
            f"| flip_y={r.get('flip_y_used', False)} | no_toc_structured={r.get('structured_from_text_headings', False)}"
        )
    except Exception as e:
        failed += 1
        print(f"❌ FAILED {s.get('path')}: {e}")

summary_path = PRE_RAG_DIR / "pdf_extraction_summary.json"
summary_path.write_text(
    json.dumps({"unit": UNIT_CODE, "results": results, "skipped_non_pdf": skipped, "failed": failed}, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("-" * 70)
print(f"Done. Extracted PDFs: {len(results)} | Failed: {failed} | Skipped non-PDF: {skipped}")
print(f"Summary: {summary_path}")
print(f"Per-source outputs: {PRE_RAG_DIR}")


Loaded 4 sources from: data_output/MBAI5004/sources_manifest.json
✅ Ethical AI in Information Technology Navigating Bias, Privacy, Transparency,.pdf | wrote=97 | warnings=29 | flip_y=False | no_toc_structured=True
✅ Ethics Of Artificial Intelligence -- S_ Matthew Liao.pdf | wrote=225 | warnings=17 | flip_y=False | no_toc_structured=False
✅ Ethics of Artificial Intelligence Case Studies.pdf | wrote=90 | warnings=8 | flip_y=True | no_toc_structured=False
✅ Ethics, Governance, and Policies in Artificial Intelligence -- Luciano Floridi.pdf | wrote=260 | warnings=34 | flip_y=True | no_toc_structured=False
----------------------------------------------------------------------
Done. Extracted PDFs: 4 | Failed: 0 | Skipped non-PDF: 0
Summary: data_output/MBAI5004/pre_rag_pdf_sections/pdf_extraction_summary.json
Per-source outputs: data_output/MBAI5004/pre_rag_pdf_sections


# Rag After Text extraction by sections